# Label-conditional conformal prediction for multi-label 12-lead ECG classification with cross-institution validation

**Research question.** Can a 12-lead ECG classifier trained on PTB-XL provide *per-diagnosis* uncertainty guarantees that
(i) hold in-domain for both positive and negative cases of every diagnostic superclass, (ii) remain valid across sex and age
subgroups, and (iii) can be restored at an unseen hospital with a small, quantified budget of labelled local ECGs?

**Surviving gap.** PTB-XL benchmarks report discrimination (AUROC) almost exclusively. Uncertainty work on ECGs typically reports
marginal (pooled) calibration or coverage, which can hide systematic under-coverage of the *positive* (diseased) class of rare
labels, and is rarely re-tested on an external institution. Nobody answers the deployment question a hospital actually faces:
*how many labelled local ECGs are needed before a stated rule-out guarantee is valid again?*

**Method.**
1. Backbone: 1D residual network on 10-s, 100-Hz, 12-lead ECGs; 5-seed deep ensemble (logit averaging).
2. Per-label temperature scaling fitted on a patient-disjoint tuning split.
3. Label-conditional (Mondrian, stratified by label *and* true class) split conformal prediction on a separate patient-disjoint
   calibration split. For every superclass this gives finite-sample coverage separately for diseased and non-diseased ECGs,
   i.e. a sensitivity-type "rule-out" guarantee for positives.
4. Site adaptation: re-estimate conformal thresholds from a small labelled sample of the new site (budget n = 100 to 1000),
   compared with source-only, pooled, covariate-shift-weighted (unlabelled) and hybrid alternatives.

**Why each component.** Temperature scaling repairs probability calibration without changing ranking (AUROC). Conformal
calibration turns scores into sets with distribution-free guarantees. Mondrian stratification is what makes the guarantee hold
for the positive class of each label rather than only on average. Site-level recalibration is needed because conformal validity
assumes exchangeability, which cross-institution shift breaks.

**Hypotheses (predeclared).**
- H1: ResNet1d ensemble macro-AUROC on PTB-XL fold 10 is >= 0.90 and exceeds the best classical feature baseline (paired, patient-cluster bootstrap, Holm-adjusted).
- H2: Temperature scaling lowers macro classwise ECE to <= 0.03 without loss of AUROC.
- H3: Mondrian conformal coverage is >= 1 - alpha - 0.02 in every (label, class) stratum in-domain; global per-label conformal does not guarantee this.
- H4: Positive-class coverage stays >= 1 - alpha - 0.05 within sex and age subgroups.
- H5: Source-calibrated conformal sets lose coverage on external sites; target recalibration with n = 250 labelled ECGs restores positive-class coverage to >= 1 - alpha - 0.03.

### Novelty-to-evidence matrix
| Claim | Experiment | Metric | Table | Acceptance criterion |
|---|---|---|---|---|
| Competitive discrimination | PTB-XL fold 10, all models | macro-AUROC, AUPRC, F1 | `T02_discrimination_indomain.csv`, `T04_paired_tests.csv` | G1 |
| Calibration | Temperature vs none | classwise ECE, Brier | `T03_calibration_indomain.csv` | G2 |
| Stratum-wise guarantee | Mondrian vs global CP | coverage per (label, class), set size | `T05_conformal_indomain.csv` | G3 |
| Subgroup validity | sex, age bands | positive coverage | `T06_subgroup_coverage.csv` | G4 |
| Shift + restoration | Georgia, Chapman-Shaoxing | coverage vs labelled budget | `T09_external_conformal_summary.csv` | G5 |
| Robustness | noise, wander, lead loss, gain | AUROC, coverage | `T11_robustness.csv` | G6 |
| Deployability | latency, size | ms/ECG, MB | `T12_efficiency.csv` | G7 |

**Baselines.** Prevalence (naive); logistic regression and histogram gradient boosting on 184 hand-crafted ECG features (strong
classical); fully convolutional network (DL baseline). **Ablations** (one component each): no temperature scaling; global instead
of Mondrian conformal; single model instead of ensemble; FCN instead of ResNet backbone; each site-adaptation variant.

**External validation.** Georgia (Emory) and Chapman-Shaoxing cohorts of the PhysioNet/CinC Challenge 2021, mapped from SNOMED CT to
PTB-XL superclasses. The mapping is itself validated on the Challenge copy of PTB-XL against the original PTB-XL labels.

**Threats to validity.** Label mapping between annotation systems; no patient identifiers in the Challenge cohorts (record-level
resampling); PTB-XL fold 10 is higher-quality than folds 1 to 9; MI is too rare in both external cohorts to evaluate there.

**Compute estimate (Colab T4).** SMOKE_TEST about 15 to 25 min (mostly downloads). STANDARD_RUN about 1 to 1.5 h. FULL_PAPER_RUN about
2 to 3.5 h (downloads 30 to 60 min, training about 45 to 70 min). RAM peak about 6 GB; disk about 10 GB.

> **Reproducibility warning.** Only `FULL_PAPER_RUN` output may be used in a manuscript. `SMOKE_TEST` uses a small real subset and one
> seed and is **NOT FOR PUBLICATION**; `STANDARD_RUN` is for checking behaviour before the full run. The notebook never generates
> synthetic ECGs: if any download or integrity check fails it stops with a diagnostic. Run top-to-bottom on a fresh runtime
> (Runtime > Run all). Recommended runtime: **GPU (T4), standard RAM**.

In [ ]:
# ============================== CONFIGURATION ==============================
MODE = "SMOKE_TEST"          # "SMOKE_TEST" | "STANDARD_RUN" | "FULL_PAPER_RUN"
USE_DRIVE_CACHE = False      # True (recommended on Colab): keep downloads, processed data, models and results on Google Drive,
                             # so a disconnect never loses work; "Run all" again resumes where it stopped
PERSIST_DIR = None           # advanced: any folder that survives runtime resets (set automatically when USE_DRIVE_CACHE)
PROJECT_NAME = "ECG-LCCP"

PTBXL_VERSION = "1.0.3"
PTBXL_BASE_URL = "https://physionet.org/files/ptb-xl/1.0.3/"
PTBXL_MIRROR_URL = "https://physionet-open.s3.amazonaws.com/ptb-xl/1.0.3/"   # PhysioNet open-data mirror on AWS (same files)
PTBXL_ZIP_URL = "https://physionet.org/static/published-projects/ptb-xl/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3.zip"
CHALLENGE_VERSION = "1.0.3"
CHALLENGE_BASE_URL = "https://physionet.org/files/challenge-2021/1.0.3/"
SNOMED_MAP_URLS = {
    "scored": "https://raw.githubusercontent.com/physionetchallenges/evaluation-2021/main/dx_mapping_scored.csv",
    "unscored": "https://raw.githubusercontent.com/physionetchallenges/evaluation-2021/main/dx_mapping_unscored.csv",
}
EXTERNAL_SITES = {"georgia": "georgia", "chapman": "chapman_shaoxing"}
AUDIT_SITE = "ptb-xl"   # Challenge copy of PTB-XL, used only to validate the SNOMED-to-superclass mapping

CLASSES = ["NORM", "MI", "STTC", "CD", "HYP"]
LEADS = ["I", "II", "III", "AVR", "AVL", "AVF", "V1", "V2", "V3", "V4", "V5", "V6"]
TARGET_FS = 100
WINDOW_S = 10
HIGHPASS_HZ = 0.5
CLIP_MV = 20.0

ALPHA_PRIMARY = 0.10
N_MIN_STRATUM_HYBRID = 20     # hybrid site adaptation: minimum labelled target cases per (label, class) stratum
MIN_POS_EXTERNAL = 100        # a class is evaluable at an external site only with >= this many positives and negatives
KAPPA_MIN_MAPPING = 0.60      # mapped classes below this agreement are reported as exploratory

MODE_CONFIG = {
    "SMOKE_TEST": dict(ptbxl_download="mirror", ptbxl_per_fold=80, ext_subdirs=1, ext_max_records=300,
                       audit_subdirs=1, audit_max_records=300, seeds=[0], archs=["resnet1d", "fcn"],
                       epochs=3, patience=2, batch_size=64, n_boot=100, n_repeats=2, n_adapt=[50],
                       n_unlabelled=100, alphas=[0.10], min_pos_ext=5, robustness=True, hgb_seeds=[0]),
    "STANDARD_RUN": dict(ptbxl_download="mirror", ptbxl_per_fold=None, ext_subdirs=None, ext_max_records=2000,
                         audit_subdirs=None, audit_max_records=2000, seeds=[0, 1, 2], archs=["resnet1d", "fcn"],
                         epochs=25, patience=5, batch_size=128, n_boot=500, n_repeats=10, n_adapt=[100, 250, 500],
                         n_unlabelled=1000, alphas=[0.10], min_pos_ext=MIN_POS_EXTERNAL, robustness=True, hgb_seeds=[0, 1, 2]),
    "FULL_PAPER_RUN": dict(ptbxl_download="mirror", ptbxl_per_fold=None, ext_subdirs=None, ext_max_records=5000,
                           audit_subdirs=None, audit_max_records=5000, seeds=[0, 1, 2, 3, 4], archs=["resnet1d", "fcn"],
                           epochs=40, patience=6, batch_size=128, n_boot=1000, n_repeats=20, n_adapt=[100, 250, 500, 1000],
                           n_unlabelled=1000, alphas=[0.10, 0.05], min_pos_ext=MIN_POS_EXTERNAL, robustness=True,
                           hgb_seeds=[0, 1, 2, 3, 4]),
}
assert MODE in MODE_CONFIG, f"MODE must be one of {list(MODE_CONFIG)}; got {MODE!r}"
CFG = dict(MODE_CONFIG[MODE])
EXTERNAL_SAMPLE_SEED = 2021   # fixed seed for the random record sample per external site (ext_max_records)
MIRROR_WORKERS = 32          # the AWS mirror handles many parallel connections
DOWNLOAD_WORKERS = 6          # PhysioNet throttles many parallel connections from one IP; 4-6 is reliable
GLOBAL_SEED = 20260929
print(f"MODE = {MODE}  |  manuscript eligible: {MODE == 'FULL_PAPER_RUN'} (final eligibility also requires verified data)")
print(CFG)

In [ ]:
# ============================== SETUP: imports, versions, determinism ==============================
import os, sys, platform, time, json, math, hashlib, random, re, ast, shutil, zipfile, io, warnings, traceback, datetime
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
os.environ.setdefault("PYTHONHASHSEED", "0")
from pathlib import Path
from fractions import Fraction
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
import scipy
from scipy import signal as sps
from scipy import stats as sstats
from scipy import io as sio
from scipy.optimize import minimize_scalar
import sklearn
from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score, balanced_accuracy_score,
                             cohen_kappa_score)
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedKFold, cross_val_predict
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)
import logging
logging.getLogger("urllib3").setLevel(logging.ERROR)   # retries are handled and summarised by the notebook
warnings.filterwarnings("ignore", message=".*padding=.same.*")
pd.set_option("display.width", 200)

IN_COLAB = "google.colab" in sys.modules
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def _ver(v):
    return tuple(int(x) for x in re.findall(r"\d+", v)[:2])

REQUIRED = {"numpy": (np.__version__, (1, 23)), "pandas": (pd.__version__, (1, 5)), "scipy": (scipy.__version__, (1, 9)),
            "scikit-learn": (sklearn.__version__, (1, 2)), "torch": (torch.__version__, (2, 0)),
            "requests": (requests.__version__, (2, 25)), "matplotlib": (matplotlib.__version__, (3, 5))}
problems = [f"{k} {v} < {'.'.join(map(str, m))}" for k, (v, m) in REQUIRED.items() if _ver(v) < m]
if problems:
    raise RuntimeError("Package versions too old: " + "; ".join(problems) +
                       ". Why: APIs used here (torch.amp, sklearn HistGradientBoosting) need newer releases. "
                       "Fix: use a current Colab runtime (Runtime > Disconnect and delete runtime, then reconnect).")
for fn_name, obj in [("torch.amp.autocast", getattr(torch, "amp", None)), ("scipy.signal.sosfiltfilt", sps),
                     ("scipy.signal.resample_poly", sps), ("scipy.stats.binomtest", sstats)]:
    attr = fn_name.split(".")[-1]
    if obj is None or not hasattr(obj, attr):
        raise RuntimeError(f"{fn_name} not available in this environment. Fix: use a current Colab runtime.")
print("No package installation is required: every dependency ships with the standard Colab runtime.")

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(GLOBAL_SEED)
try:
    torch.use_deterministic_algorithms(True, warn_only=True)
except Exception as e:
    print("Deterministic algorithms not fully available:", e)

RUNTIME_INFO = {
    "timestamp_utc": datetime.datetime.now(datetime.timezone.utc).replace(tzinfo=None).isoformat(timespec="seconds") + "Z",
    "mode": MODE, "python": sys.version.split()[0], "platform": platform.platform(), "in_colab": IN_COLAB,
    "device": str(DEVICE), "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "cpu_count": os.cpu_count(),
    "versions": {k: v for k, (v, _) in REQUIRED.items()},
}
print(json.dumps(RUNTIME_INFO, indent=1))
if MODE != "SMOKE_TEST" and DEVICE.type != "cuda":
    print("WARNING: no GPU detected. Training will be very slow. Fix: Runtime > Change runtime type > T4 GPU.")

In [ ]:
# ============================== PATHS, CACHE, OUTPUTS ==============================
if IN_COLAB:
    WORK = Path("/content/cardiotrust")
else:
    WORK = Path.cwd() / "cardiotrust_work"
import tarfile
if USE_DRIVE_CACHE and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PERSIST_DIR = "/content/drive/MyDrive/ECG_LCCP_run"
PERSIST = Path(PERSIST_DIR) if PERSIST_DIR else None
RAW = WORK / "raw"                       # raw downloads always go to the fast local disk ...
SHARDS = PERSIST / "raw_shards" if PERSIST else None   # ... and are packed into archives on Drive every 2,000 files
CACHE = (PERSIST or WORK) / "cache"      # processed signals and features
OUT = (PERSIST or WORK) / "project_outputs"
SUB = {k: OUT / k for k in ["config", "data_audit", "predictions", "tables", "figures", "models", "logs"]}
SUB["checkpoints"] = WORK / "checkpoints"   # per-epoch training state stays local (large, rewritten every epoch)
for p in [RAW, CACHE, OUT, *SUB.values()] + ([SHARDS] if SHARDS else []):
    p.mkdir(parents=True, exist_ok=True)

def restore_shards():
    """After a disconnect: unpack previously saved download archives from Drive onto the local disk."""
    if not SHARDS:
        return 0
    n_files, t0 = 0, time.time()
    for tgz in sorted(SHARDS.glob("*.tar.gz")):
        try:
            with tarfile.open(tgz, "r:gz") as tf:
                members = [m for m in tf.getmembers() if m.isfile() and not (RAW / m.name).exists()]
                for m in members:
                    if m.name.startswith("/") or ".." in Path(m.name).parts:
                        continue
                    tf.extract(m, RAW)
                n_files += len(members)
        except (tarfile.TarError, OSError, EOFError) as e:
            print(f"  skipping damaged archive {tgz.name} ({e}); its files will be downloaded again")
    if n_files:
        print(f"Restored {n_files} previously downloaded files from Drive in {time.time()-t0:.0f}s")
    return n_files

def persist_files(paths, tag):
    """Pack newly downloaded files into one compressed archive on Drive (fast; one Drive write instead of thousands)."""
    if not SHARDS or not paths:
        return
    name = f"{re.sub(r'[^A-Za-z0-9]+', '_', tag)[:40]}_{datetime.datetime.now().strftime('%Y%m%d%H%M%S%f')}.tar.gz"
    tmp = WORK / name
    with tarfile.open(tmp, "w:gz", compresslevel=1) as tf:
        for p in paths:
            tf.add(p, arcname=str(Path(p).relative_to(RAW)))
    part = SHARDS / (name + ".part")
    shutil.copyfile(tmp, part)
    part.rename(SHARDS / name)
    tmp.unlink()

RESTORED_FILES = restore_shards()
LOG_FILE = SUB["logs"] / "run_log.txt"

def log(msg):
    line = f"[{datetime.datetime.now(datetime.timezone.utc).replace(tzinfo=None).strftime('%H:%M:%S')}] {msg}"
    print(line)
    with open(LOG_FILE, "a") as fh:
        fh.write(line + "\n")

def _report_save(path):
    size = path.stat().st_size
    if size == 0:
        raise RuntimeError(f"Saved file is empty: {path}")
    return size

def save_table(df, name, index=False):
    path = SUB["tables"] / f"{name}.csv"
    df.to_csv(path, index=index)
    size = _report_save(path)
    print(f"  saved table {path.relative_to(OUT)}  shape={df.shape}  {size/1024:.1f} KB")
    return path

def save_json(obj, subdir, name):
    path = SUB[subdir] / f"{name}.json"
    with open(path, "w") as fh:
        json.dump(obj, fh, indent=1, default=lambda o: o.item() if hasattr(o, "item") else str(o))
    size = _report_save(path)
    print(f"  saved {path.relative_to(OUT)}  {size/1024:.1f} KB")
    return path

def save_fig(fig, name):
    paths = []
    for ext in ["pdf", "svg", "png"]:
        p = SUB["figures"] / f"{name}.{ext}"
        fig.savefig(p, dpi=300, bbox_inches="tight")
        _report_save(p)
        paths.append(p)
    plt.close(fig)
    print(f"  saved figure {name}.pdf/.svg/.png (300 dpi)")
    return paths

save_json({"mode": MODE, "config": CFG, "runtime": RUNTIME_INFO, "classes": CLASSES, "leads": LEADS,
           "target_fs": TARGET_FS, "window_s": WINDOW_S, "highpass_hz": HIGHPASS_HZ, "clip_mv": CLIP_MV,
           "alpha_primary": ALPHA_PRIMARY, "n_min_stratum_hybrid": N_MIN_STRATUM_HYBRID,
           "kappa_min_mapping": KAPPA_MIN_MAPPING, "global_seed": GLOBAL_SEED,
           "urls": {"ptbxl": PTBXL_BASE_URL, "ptbxl_zip": PTBXL_ZIP_URL, "challenge": CHALLENGE_BASE_URL,
                    "snomed": SNOMED_MAP_URLS}}, "config", "run_config")
log(f"Work dir {WORK}; outputs {OUT}; raw {RAW}; persistent store {PERSIST or 'none (local only)'}")

In [ ]:
# ============================== HTTP: preflight, retries, integrity ==============================
class DataAcquisitionError(RuntimeError):
    pass

def make_session():
    s = requests.Session()
    retry = Retry(total=6, backoff_factor=1.5, status_forcelist=[429, 500, 502, 503, 504],
                  allowed_methods=["GET", "HEAD"], raise_on_status=False)
    adapter = HTTPAdapter(max_retries=retry, pool_connections=8, pool_maxsize=64)
    s.mount("https://", adapter); s.mount("http://", adapter)
    s.headers.update({"User-Agent": "ECG-LCCP-research-notebook/1.0 (academic use)"})
    return s

SESSION = make_session()

def looks_like_html(b):
    head = b[:1024].lstrip().lower()
    return head.startswith(b"<!doctype html") or head.startswith(b"<html") or b"<head>" in head[:300]

def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for block in iter(lambda: fh.read(chunk), b""):
            h.update(block)
    return h.hexdigest()

def fetch_bytes(url, min_bytes=1, allow_html=False, timeout=(30, 180)):
    try:
        r = SESSION.get(url, timeout=timeout)
    except requests.RequestException as e:
        raise DataAcquisitionError(f"Network error for {url}: {e}. Why: server unreachable or connection reset. "
                                   f"Fix: re-run this cell; if it persists check https://physionet.org status.") from e
    if r.status_code != 200:
        raise DataAcquisitionError(f"HTTP {r.status_code} for {url}. Why: file moved, rate limiting or server error. "
                                   f"Fix: re-run later; confirm the URL in a browser.")
    content = r.content
    if len(content) < min_bytes:
        raise DataAcquisitionError(f"Empty/short response ({len(content)} bytes) for {url}.")
    if not allow_html and looks_like_html(content):
        raise DataAcquisitionError(f"Got an HTML page instead of data for {url}. Why: login wall, error page or wrong path. "
                                   f"Fix: open the URL in a browser to check.")
    return content

def download_file(url, dest, expected_sha=None, min_bytes=1, persist=True):
    """Atomic download with HTML detection and optional SHA-256 check. Returns 'cached' or 'downloaded'."""
    dest = Path(dest)
    if dest.exists() and dest.stat().st_size >= min_bytes:
        if expected_sha is None or sha256_file(dest) == expected_sha:
            return "cached"
    dest.parent.mkdir(parents=True, exist_ok=True)
    content = fetch_bytes(url, min_bytes=min_bytes)
    if expected_sha is not None:
        got = hashlib.sha256(content).hexdigest()
        if got != expected_sha:
            raise DataAcquisitionError(f"Checksum mismatch for {url}: expected {expected_sha[:12]}..., got {got[:12]}... "
                                       f"Why: corrupted or changed file. Fix: delete {dest} and re-run.")
    tmp = dest.with_suffix(dest.suffix + ".part")
    with open(tmp, "wb") as fh:
        fh.write(content)
    tmp.replace(dest)
    if persist and SHARDS and Path(dest).is_relative_to(RAW):
        persist_files([dest], "meta_" + Path(dest).name)   # small metadata files (checksum lists, indexes) survive restarts too
    return "downloaded"

def download_large(url, dest, expected_sha=None):
    dest = Path(dest)
    if dest.exists() and dest.stat().st_size > 0 and (expected_sha is None or sha256_file(dest) == expected_sha):
        log(f"cached: {dest.name} ({dest.stat().st_size/1e9:.2f} GB)")
        return "cached"
    tmp = dest.with_suffix(dest.suffix + ".part")
    t0 = time.time()
    with SESSION.get(url, stream=True, timeout=(15, 300), allow_redirects=True) as r:
        if r.status_code != 200:
            raise DataAcquisitionError(f"HTTP {r.status_code} for {url}. Fix: retry later or switch to per-file download.")
        ctype = r.headers.get("Content-Type", "")
        if "html" in ctype.lower():
            raise DataAcquisitionError(f"Expected a ZIP from {url} but got Content-Type {ctype}.")
        total = int(r.headers.get("Content-Length", 0))
        done, last_print = 0, time.time()
        with open(tmp, "wb") as fh:
            for block in r.iter_content(chunk_size=1 << 22):
                fh.write(block); done += len(block)
                if time.time() - last_print >= 30:
                    el = time.time() - t0; rate = done / 1e6 / el
                    eta = f", ~{(total - done) / 1e6 / max(rate, 1e-6) / 60:.0f} min left" if total else ""
                    print(f"  {dest.name}: {done/1e9:.2f}" + (f" / {total/1e9:.2f}" if total else "") + f" GB, {rate:.1f} MB/s{eta}")
                    last_print = time.time()
    if total and tmp.stat().st_size != total:
        raise DataAcquisitionError(f"Incomplete download for {url}: {tmp.stat().st_size} of {total} bytes. Fix: re-run.")
    tmp.replace(dest)
    if expected_sha is not None and sha256_file(dest) != expected_sha:
        raise DataAcquisitionError(f"Checksum mismatch for {dest}. Fix: delete it and re-run.")
    log(f"downloaded {dest.name} in {time.time()-t0:.0f}s")
    return "downloaded"

def parse_sha256sums(text):
    out = {}
    for line in text.splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        parts = line.split(maxsplit=1)
        if len(parts) != 2 or not re.fullmatch(r"[0-9a-fA-F]{64}", parts[0]):
            continue
        out[parts[1].strip().lstrip("*")] = parts[0].lower()
    return out

def parallel_download(jobs, desc, workers=None, chunk=2000):
    """jobs: list of (url, dest, sha). Downloads in chunks; after each chunk the new files are archived to Drive
    (when enabled), so a disconnect loses at most one chunk. Raises with a summary if files still fail."""
    counts = {"cached": 0, "downloaded": 0}
    failures = []
    t0, done = time.time(), 0

    def run(batch, n_workers):
        new_files, fails = [], []
        with ThreadPoolExecutor(max_workers=n_workers) as ex:
            futs = {ex.submit(download_file, u, d, s, 1, False): (u, d) for (u, d, s) in batch}
            for fut in as_completed(futs):
                u, d = futs[fut]
                try:
                    r = fut.result()
                    counts[r] += 1
                    if r == "downloaded":
                        new_files.append(d)
                except Exception as e:
                    fails.append((u, str(e)[:200]))
        persist_files(new_files, desc)
        return fails

    for start in range(0, len(jobs), chunk):
        batch = jobs[start:start + chunk]
        failures += run(batch, workers or DOWNLOAD_WORKERS)
        done += len(batch)
        el = time.time() - t0
        print(f"  {desc}: {done}/{len(jobs)} files ({el:.0f}s, {done/max(el,1e-6):.1f} files/s, "
              f"~{(len(jobs)-done)/max(done/max(el,1e-6),1e-6)/60:.0f} min left)" + ("  [saved to Drive]" if SHARDS else ""))
    if failures:
        # Second, gentler pass: 2 connections with a pause, for files that failed under load
        log(f"{desc}: {len(failures)} files failed on the first pass; retrying slowly with 2 connections")
        time.sleep(30)
        failed_urls = {u for u, _ in failures}
        failures = run([j for j in jobs if j[0] in failed_urls], 2)
    if failures:
        raise DataAcquisitionError(f"{len(failures)} of {len(jobs)} downloads still failed for {desc}. First: {failures[:2]}. "
                                   f"Why: PhysioNet is throttling or temporarily unreachable. Fix: wait 10-15 min and re-run "
                                   f"this cell; completed files are cached and verified, so only the missing ones are fetched.")
    return counts

PROVENANCE = {"accessed_utc": RUNTIME_INFO["timestamp_utc"], "sources": {}}

## Dataset citation and acquisition
- **PTB-XL v1.0.3** (PhysioNet, CC BY 4.0, DOI 10.13026/kfzx-aw45; Wagner et al., *Sci Data* 2020, 10.1038/s41597-020-0495-6).
  21,799 10-s 12-lead ECGs from 18,869 patients; patient-stratified folds 1 to 10 (fold 10 is the recommended test fold).
- **PhysioNet/CinC Challenge 2021 v1.0.3** (PhysioNet, CC BY 4.0, DOI 10.13026/34va-7q14; Reyna et al., *CinC* 2021,
  10.23919/CinC53138.2021.9662687). External cohorts: `training/georgia` (10,344 ECGs) and `training/chapman_shaoxing`
  (10,247 ECGs); `training/ptb-xl` headers are used only to validate the label mapping.
- **SNOMED CT label tables** `dx_mapping_scored.csv` / `dx_mapping_unscored.csv` from the official Challenge evaluation repository.

Access order: PhysioNet direct download (ZIP for full PTB-XL; per-file otherwise). Every file is verified against PhysioNet's
`SHA256SUMS.txt`. Nothing is simulated.

In [ ]:
# ---------- PTB-XL ----------
PTBXL_DIR = RAW / "ptbxl"
PTBXL_DIR.mkdir(parents=True, exist_ok=True)

# Preflight: tiny request that must return plain-text checksums, not HTML
sha_txt = PTBXL_DIR / "SHA256SUMS.txt"
download_file(PTBXL_BASE_URL + "SHA256SUMS.txt", sha_txt, min_bytes=1000)
PTBXL_SHA = parse_sha256sums(sha_txt.read_text())
if len(PTBXL_SHA) < 1000:
    raise DataAcquisitionError(f"PTB-XL SHA256SUMS parsed only {len(PTBXL_SHA)} entries. Fix: delete {sha_txt} and re-run.")
for fname in ["ptbxl_database.csv", "scp_statements.csv"]:
    download_file(PTBXL_BASE_URL + fname, PTBXL_DIR / fname, expected_sha=PTBXL_SHA.get(fname), min_bytes=100)

ptbxl_db = pd.read_csv(PTBXL_DIR / "ptbxl_database.csv", index_col="ecg_id")
scp_df = pd.read_csv(PTBXL_DIR / "scp_statements.csv", index_col=0)
need_cols = {"patient_id", "age", "sex", "scp_codes", "strat_fold", "filename_hr"}
missing = need_cols - set(ptbxl_db.columns)
if missing:
    raise DataAcquisitionError(f"ptbxl_database.csv lacks columns {missing}. Why: schema change. Fix: check PTB-XL version.")
if not {"diagnostic", "diagnostic_class"} <= set(scp_df.columns):
    raise DataAcquisitionError("scp_statements.csv lacks diagnostic columns.")
print(f"ptbxl_database.csv: {len(ptbxl_db)} records, {ptbxl_db.patient_id.nunique()} patients")

# Select records (SMOKE: a small fold-stratified real subset)
if CFG["ptbxl_per_fold"]:
    rng = np.random.default_rng(GLOBAL_SEED)
    sel = []
    for fold, grp in ptbxl_db.groupby("strat_fold"):
        pats = grp.patient_id.unique()
        rng.shuffle(pats)
        chosen = grp[grp.patient_id.isin(pats[:CFG["ptbxl_per_fold"]])].index.tolist()
        sel.extend(chosen)
    ptbxl_sel = ptbxl_db.loc[sorted(sel)].copy()
else:
    ptbxl_sel = ptbxl_db.copy()
print(f"Selected PTB-XL records: {len(ptbxl_sel)}")

rec_files = []
for fn in ptbxl_sel.filename_hr:
    rec_files += [fn + ".hea", fn + ".dat"]

PTBXL_ROUTE = CFG["ptbxl_download"]
if PTBXL_ROUTE == "mirror":
    try:
        jobs = [(PTBXL_MIRROR_URL + f, PTBXL_DIR / f, PTBXL_SHA.get(f)) for f in rec_files]
        counts = parallel_download(jobs, "PTB-XL records (PhysioNet AWS open-data mirror)", workers=MIRROR_WORKERS)
    except DataAcquisitionError as e:
        log(f"Mirror route failed ({str(e)[:200]}); falling back to the PhysioNet ZIP.")
        PTBXL_ROUTE = "zip"
if PTBXL_ROUTE == "zip":
    zpath = RAW / "ptbxl_1.0.3.zip"
    extracted_marker = PTBXL_DIR / ".zip_extracted"
    if not extracted_marker.exists():
        try:
            download_large(PTBXL_ZIP_URL, zpath)
            with zipfile.ZipFile(zpath) as zf:
                names = zf.namelist()
                root = names[0].split("/")[0]
                members = [n for n in names if "/records500/" in n and not n.endswith("/")]
                log(f"Extracting {len(members)} 500-Hz files from ZIP root '{root}'")
                for n in members:
                    rel = n.split("/", 1)[1]
                    target = PTBXL_DIR / rel
                    target.parent.mkdir(parents=True, exist_ok=True)
                    with zf.open(n) as src, open(target, "wb") as dst:
                        shutil.copyfileobj(src, dst)
            extracted_marker.write_text("ok")
            zpath.unlink(missing_ok=True)
        except (DataAcquisitionError, zipfile.BadZipFile, requests.RequestException) as e:
            log(f"ZIP route failed ({e}); falling back to per-file download (slower, same checksums).")
    # Verify (and fetch anything missing) against SHA256SUMS
    jobs = [(PTBXL_BASE_URL + f, PTBXL_DIR / f, PTBXL_SHA.get(f)) for f in rec_files]
    counts = parallel_download(jobs, "PTB-XL verify/fetch")
elif PTBXL_ROUTE == "files":
    jobs = [(PTBXL_BASE_URL + f, PTBXL_DIR / f, PTBXL_SHA.get(f)) for f in rec_files]
    counts = parallel_download(jobs, "PTB-XL records")
n_without_sha = sum(PTBXL_SHA.get(f) is None for f in rec_files)
PROVENANCE["sources"]["PTB-XL"] = {
    "name": "PTB-XL, a large publicly available electrocardiography dataset", "version": PTBXL_VERSION,
    "url": PTBXL_BASE_URL, "doi": "10.13026/kfzx-aw45", "license": "CC BY 4.0",
    "download_route": PTBXL_ROUTE, "mirror_url": PTBXL_MIRROR_URL if PTBXL_ROUTE == "mirror" else None,
    "checksums_from": PTBXL_BASE_URL + "SHA256SUMS.txt",
    "files_used": len(rec_files), "files_without_checksum": int(n_without_sha), "download_counts": counts,
    "records_selected": int(len(ptbxl_sel)), "patients_selected": int(ptbxl_sel.patient_id.nunique()),
    "sha256_database_csv": sha256_file(PTBXL_DIR / "ptbxl_database.csv")}
if n_without_sha:
    raise DataAcquisitionError(f"{n_without_sha} PTB-XL files have no checksum entry. Fix: re-download SHA256SUMS.txt.")
log(f"PTB-XL ready: {counts}")

In [ ]:
# ---------- PhysioNet/CinC Challenge 2021 (external sites + mapping audit) ----------
CH_DIR = RAW / "challenge2021"
CH_DIR.mkdir(parents=True, exist_ok=True)
download_file(CHALLENGE_BASE_URL + "RECORDS", CH_DIR / "RECORDS", min_bytes=50)
ch_sha_path = CH_DIR / "SHA256SUMS.txt"
download_file(CHALLENGE_BASE_URL + "SHA256SUMS.txt", ch_sha_path, min_bytes=1000)
CH_SHA = parse_sha256sums(ch_sha_path.read_text())
if len(CH_SHA) < 1000:
    raise DataAcquisitionError(f"Challenge SHA256SUMS parsed only {len(CH_SHA)} entries. Fix: delete {ch_sha_path}, re-run.")
top_dirs = [l.strip().rstrip("/") for l in (CH_DIR / "RECORDS").read_text().splitlines() if l.strip()]

def natural_key(s):
    return [int(t) if t.isdigit() else t for t in re.split(r"(\d+)", s)]

def site_subdirs(site_folder, n_subdirs):
    subs = sorted([d for d in top_dirs if d.startswith(f"training/{site_folder}/")], key=natural_key)
    if not subs:
        raise DataAcquisitionError(f"No subfolders for '{site_folder}' in Challenge RECORDS. Why: folder renamed. "
                                   f"Fix: inspect {CHALLENGE_BASE_URL}RECORDS.")
    return subs if n_subdirs is None else subs[:n_subdirs]

UNPUBLISHED_DROPPED = {}

def site_records(site_folder, n_subdirs, max_records):
    recs = []
    for sd in site_subdirs(site_folder, n_subdirs):
        p = CH_DIR / sd / "RECORDS"
        download_file(CHALLENGE_BASE_URL + sd + "/RECORDS", p, min_bytes=5)
        names = [l.strip() for l in p.read_text().splitlines() if l.strip()]
        recs += [f"{sd}/{n}" for n in names]
    # PhysioNet's per-folder RECORDS lists include some records that were never published (e.g. georgia/g3/E03000 -> 404).
    # SHA256SUMS.txt is the authoritative inventory: keep only records whose header (and signal) files actually exist.
    listed = len(recs)
    recs = [r for r in recs if (r + ".hea") in CH_SHA and (site_folder == AUDIT_SITE or (r + ".mat") in CH_SHA)]
    UNPUBLISHED_DROPPED[site_folder] = listed - len(recs)
    if listed - len(recs):
        log(f"{site_folder}: {listed - len(recs)} of {listed} listed records are not published on PhysioNet (no files); excluded")
    if max_records and len(recs) > max_records:
        # fixed-seed uniform random sample across all selected subfolders (predeclared; avoids record-ID/time ordering bias)
        rng_s = np.random.default_rng(EXTERNAL_SAMPLE_SEED)
        recs = sorted(rng_s.choice(recs, size=max_records, replace=False).tolist(), key=natural_key)
    return recs

def fetch_challenge(recs, with_signal, desc):
    exts = [".hea", ".mat"] if with_signal else [".hea"]
    jobs = []
    for r in recs:
        for ext in exts:
            rel = r + ext
            sha = CH_SHA.get(rel)
            if sha is None:
                raise DataAcquisitionError(f"No checksum for {rel}. Fix: re-download SHA256SUMS.txt.")
            jobs.append((CHALLENGE_BASE_URL + rel, CH_DIR / rel, sha))
    return parallel_download(jobs, desc)

EXT_RECORDS = {}
for key, folder in EXTERNAL_SITES.items():
    EXT_RECORDS[key] = site_records(folder, CFG["ext_subdirs"], CFG["ext_max_records"])
    c = fetch_challenge(EXT_RECORDS[key], True, f"external {key}")
    PROVENANCE["sources"][f"Challenge2021-{key}"] = {"folder": f"training/{folder}", "records": len(EXT_RECORDS[key]),
                                                      "sampling": (f"uniform random sample of {CFG['ext_max_records']} records "
                                                                   f"(seed {EXTERNAL_SAMPLE_SEED})" if CFG["ext_max_records"] else "all records"),
                                                      "listed_but_unpublished_excluded": UNPUBLISHED_DROPPED.get(folder, 0),
                                                      "download_counts": c}
AUDIT_RECORDS = site_records(AUDIT_SITE, CFG["audit_subdirs"], CFG["audit_max_records"])
c = fetch_challenge(AUDIT_RECORDS, False, "mapping-audit headers (ptb-xl)")
PROVENANCE["sources"]["Challenge2021-ptbxl-headers"] = {"records": len(AUDIT_RECORDS), "download_counts": c}
PROVENANCE["sources"]["Challenge2021"] = {
    "name": "Will Two Do? Varying Dimensions in Electrocardiography: The PhysioNet/CinC Challenge 2021",
    "version": CHALLENGE_VERSION, "url": CHALLENGE_BASE_URL, "doi": "10.13026/34va-7q14", "license": "CC BY 4.0"}

SNOMED = {}
for k, url in SNOMED_MAP_URLS.items():
    p = RAW / f"dx_mapping_{k}.csv"
    download_file(url, p, min_bytes=500)
    df = pd.read_csv(p)
    if not {"Dx", "SNOMEDCTCode", "Abbreviation"} <= set(df.columns):
        raise DataAcquisitionError(f"{p.name} schema changed: {list(df.columns)}")
    SNOMED[k] = df
    PROVENANCE["sources"][f"SNOMED-map-{k}"] = {"url": url, "rows": len(df), "sha256": sha256_file(p)}
snomed_all = pd.concat(SNOMED.values(), ignore_index=True)[["Dx", "SNOMEDCTCode", "Abbreviation"]]
log(f"Challenge data ready: " + ", ".join(f"{k}={len(v)}" for k, v in EXT_RECORDS.items()) + f", audit headers={len(AUDIT_RECORDS)}")

In [ ]:
# ---------- WFDB header / signal parsers (no external wfdb dependency) ----------
class HeaderParseError(ValueError):
    pass

GAIN_RE = re.compile(r"^([0-9.eE+-]+)(?:\((-?\d+)\))?(?:/(\S+))?$")

def parse_header(text):
    if text is None or not text.strip():
        raise HeaderParseError("empty header")
    if looks_like_html(text.encode()):
        raise HeaderParseError("HTML instead of a WFDB header")
    lines = [l.rstrip() for l in text.splitlines() if l.strip()]
    rec = lines[0].split()
    if len(rec) < 4:
        raise HeaderParseError(f"malformed record line: {lines[0]!r}")
    try:
        nsig, fs, nsamp = int(rec[1]), float(rec[2].split("/")[0]), int(rec[3])
    except ValueError as e:
        raise HeaderParseError(f"non-numeric record line: {lines[0]!r}") from e
    sig_lines = [l for l in lines[1:] if not l.startswith("#")][:nsig]
    if len(sig_lines) != nsig:
        raise HeaderParseError(f"expected {nsig} signal lines, found {len(sig_lines)}")
    chans = []
    for l in sig_lines:
        p = l.split()
        if len(p) < 3:
            raise HeaderParseError(f"malformed signal line: {l!r}")
        m = GAIN_RE.match(p[2])
        if not m:
            raise HeaderParseError(f"bad gain field {p[2]!r}")
        gain = float(m.group(1)) or 200.0
        adczero = int(p[4]) if len(p) > 4 and re.fullmatch(r"-?\d+", p[4]) else 0
        baseline = int(m.group(2)) if m.group(2) is not None else adczero
        units = (m.group(3) or "mV").lower()
        chans.append({"file": p[0], "fmt": p[1], "gain": gain, "baseline": baseline, "units": units,
                      "lead": p[-1].upper()})
    meta = {}
    for l in lines:
        if l.startswith("#") and ":" in l:
            k, v = l[1:].split(":", 1)
            meta[k.strip().lower()] = v.strip()
    return {"name": rec[0], "nsig": nsig, "fs": fs, "nsamp": nsamp, "channels": chans, "meta": meta}

def read_signal(hea_path):
    """Return (signal in mV with shape [n_samples, 12] in LEADS order, fs, header dict)."""
    hea_path = Path(hea_path)
    h = parse_header(hea_path.read_text())
    files = {c["file"] for c in h["channels"]}
    if len(files) != 1:
        raise HeaderParseError(f"multi-file records not supported: {files}")
    dfile = hea_path.parent / files.pop()
    if dfile.suffix == ".mat":
        raw = sio.loadmat(dfile)["val"]
        if raw.shape[0] != h["nsig"]:
            raise HeaderParseError(f"{dfile.name}: shape {raw.shape} vs nsig {h['nsig']}")
        dig = raw.T.astype(np.float64)
    else:
        if h["channels"][0]["fmt"].split("+")[0].split("x")[0] != "16":
            raise HeaderParseError(f"unsupported format {h['channels'][0]['fmt']}")
        arr = np.fromfile(dfile, dtype="<i2")
        if arr.size != h["nsig"] * h["nsamp"]:
            raise HeaderParseError(f"{dfile.name}: {arr.size} samples vs expected {h['nsig']*h['nsamp']}")
        dig = arr.reshape(h["nsamp"], h["nsig"]).astype(np.float64)
    gains = np.array([c["gain"] for c in h["channels"]])
    base = np.array([c["baseline"] for c in h["channels"]])
    phys = (dig - base) / gains
    scale = np.array([1e-3 if c["units"] in ("uv", "μv") else 1.0 for c in h["channels"]])
    phys = phys * scale
    names = [c["lead"] for c in h["channels"]]
    try:
        order = [names.index(ld) for ld in LEADS]
    except ValueError:
        raise HeaderParseError(f"leads {names} do not contain all of {LEADS}")
    return phys[:, order], h["fs"], h

SOS_HP = sps.butter(2, HIGHPASS_HZ, btype="highpass", fs=TARGET_FS, output="sos")
N_TARGET = TARGET_FS * WINDOW_S

def preprocess(sig, fs):
    """Resample to 100 Hz with anti-aliasing, keep the first 10 s, 0.5-Hz zero-phase high-pass, clip. Returns (x, status)."""
    if not np.all(np.isfinite(sig)):
        return None, "non_finite"
    frac = Fraction(TARGET_FS, int(round(fs))).limit_denominator(1000)
    x = sps.resample_poly(sig, frac.numerator, frac.denominator, axis=0) if frac != 1 else sig
    if x.shape[0] < N_TARGET:
        return None, "shorter_than_10s"
    x = sps.sosfiltfilt(SOS_HP, x[:N_TARGET], axis=0)
    n_clip = int(np.sum(np.abs(x) > CLIP_MV))
    x = np.clip(x, -CLIP_MV, CLIP_MV).astype(np.float32)
    if np.allclose(x.std(axis=0), 0):
        return None, "flat_signal"
    return x, ("ok_clipped" if n_clip else "ok")

# Parser tests: valid / empty / HTML / malformed / schema change
_valid = "X 12 500 5000\n" + "".join(f"X.mat 16x1+24 1000.0(0)/mV 16 0 0 0 0 {ld}\n" for ld in
                                     ["I", "II", "III", "aVR", "aVL", "aVF", "V1", "V2", "V3", "V4", "V5", "V6"]) + "# Age: 50\n# Sex: Male\n# Dx: 426783006\n"
_h = parse_header(_valid)
assert _h["nsig"] == 12 and _h["fs"] == 500 and _h["meta"]["dx"] == "426783006" and _h["channels"][3]["lead"] == "AVR"
for bad, why in [("", "empty"), ("<!DOCTYPE html><html><head>", "html"), ("X 12 500", "malformed"),
                 ("X 12 500 5000\nX.mat 16 1000/mV 16 0 0 0 0 I\n", "missing signal lines")]:
    try:
        parse_header(bad)
        raise AssertionError(f"parser accepted bad header ({why})")
    except HeaderParseError:
        pass
assert parse_sha256sums("a" * 64 + "  dir/f.hea\n<html>\n") == {"dir/f.hea": "a" * 64}
assert looks_like_html(b"  <!DOCTYPE html>") and not looks_like_html(b"HR00001 12 500 5000")
_t = np.linspace(0, 10, 5000, endpoint=False)
_x, _st = preprocess(np.tile(np.sin(2 * np.pi * 1.2 * _t)[:, None], (1, 12)), 500)
assert _x.shape == (N_TARGET, 12) and _st.startswith("ok")
assert preprocess(np.zeros((2000, 12)), 500)[1] == "shorter_than_10s"
print("Parser and preprocessing unit tests passed (valid, empty, HTML, malformed, short, filter shape).")

In [ ]:
# ---------- Labels: PTB-XL superclasses and SNOMED CT mapping for the Challenge cohorts ----------
diag = scp_df[scp_df.diagnostic == 1]
bad_cls = set(diag.diagnostic_class.dropna()) - set(CLASSES)
assert not bad_cls, f"Unexpected PTB-XL diagnostic classes {bad_cls}"

def ptbxl_superclasses(scp_str):
    codes = ast.literal_eval(scp_str)
    return sorted({diag.loc[c, "diagnostic_class"] for c in codes if c in diag.index})

ptbxl_sel["superclasses"] = ptbxl_sel.scp_codes.apply(ptbxl_superclasses)
n_before = len(ptbxl_sel)
ptbxl_sel = ptbxl_sel[ptbxl_sel.superclasses.str.len() > 0].copy()
Y_ptbxl = np.array([[int(c in s) for c in CLASSES] for s in ptbxl_sel.superclasses], dtype=np.int8)
log(f"PTB-XL records with >=1 diagnostic superclass: {len(ptbxl_sel)} of {n_before}")

# SNOMED CT -> PTB-XL superclass (predeclared; mirrors PTB-XL's diagnostic statements; rhythm and form codes carry no superclass)
MAP_ABBR = {
    "MI":   ["MI", "AMI", "AnMI", "OldMI"],
    "STTC": ["TAb", "TInv", "NSSTTA", "STC", "STD", "STE", "STIAb", "LQT", "AnMIs", "IIs", "LIs", "MIs", "AMIs", "CMI", "ISTD"],
    "CD":   ["RBBB", "CRBBB", "IRBBB", "LBBB", "CLBBB", "ILBBB", "BBB", "LAnFB", "LPFB", "IAVB", "IIAVB", "IIAVBII", "MoI",
             "AVB", "CHB", "NSIVCB", "DIB", "WPW"],
    "HYP":  ["LVH", "RVH", "VH", "LAE", "LAA", "LAH", "RAH", "RAAb", "RAHV", "AH"],
}
NORM_COMPATIBLE_ABBR = {"A_primary": ["NSR", "SB", "STach", "SA", "LQRSV", "LAD", "RAD"],   # sinus rhythms and axis/voltage form codes
                        "B_strict": ["NSR"]}
abbr2code = {}
for _, r in snomed_all.iterrows():
    abbr2code.setdefault(str(r.Abbreviation), []).append(int(r.SNOMEDCTCode))
map_rows = []
CLASS_CODES = {}
for cls, abbrs in MAP_ABBR.items():
    codes = set()
    for a in abbrs:
        if a not in abbr2code:
            raise DataAcquisitionError(f"Abbreviation {a} not in SNOMED tables. Why: table changed. Fix: update MAP_ABBR.")
        for code in abbr2code[a]:
            codes.add(code)
            dx = snomed_all.loc[snomed_all.SNOMEDCTCode == code, "Dx"].iloc[0]
            map_rows.append({"superclass": cls, "abbreviation": a, "snomed_ct": code, "dx": dx})
    CLASS_CODES[cls] = codes
NORM_CODES = {k: {c for a in v for c in abbr2code[a]} for k, v in NORM_COMPATIBLE_ABBR.items()}
for k, v in NORM_COMPATIBLE_ABBR.items():
    for a in v:
        for code in abbr2code[a]:
            map_rows.append({"superclass": f"NORM-compatible ({k})", "abbreviation": a, "snomed_ct": code,
                             "dx": snomed_all.loc[snomed_all.SNOMEDCTCode == code, "Dx"].iloc[0]})
save_table(pd.DataFrame(map_rows), "S01_snomed_to_superclass_map")

def map_dx(dx_codes, rule="A_primary"):
    """Return (5-vector or None, status). NORM: no class code and every code in the NORM-compatible set."""
    y = np.zeros(len(CLASSES), dtype=np.int8)
    for j, cls in enumerate(CLASSES):
        if cls != "NORM" and dx_codes & CLASS_CODES[cls]:
            y[j] = 1
    if y.sum() > 0:
        return y, "mapped"
    if dx_codes and dx_codes <= NORM_CODES[rule]:
        y[CLASSES.index("NORM")] = 1
        return y, "mapped_norm"
    return None, "indeterminate"

def header_dx(h):
    raw = h["meta"].get("dx", "")
    return {int(c) for c in re.findall(r"\d+", raw)}

def header_demo(h):
    age = pd.to_numeric(h["meta"].get("age", "nan"), errors="coerce")
    sex = h["meta"].get("sex", "").strip().lower()
    sex = 0 if sex.startswith("m") else (1 if sex.startswith("f") else -1)
    return (float(age) if np.isfinite(age) else np.nan), sex

In [ ]:
# ---------- Mapping audit: SNOMED-mapped labels vs original PTB-XL superclasses on the Challenge copy of PTB-XL ----------
audit_rows = []
for r in AUDIT_RECORDS:
    h = parse_header((CH_DIR / (r + ".hea")).read_text())
    m = re.fullmatch(r"HR(\d+)", h["name"])
    if not m:
        continue
    age, sex = header_demo(h)
    audit_rows.append({"ecg_id": int(m.group(1)), "ch_age": age, "ch_sex": sex, "dx": header_dx(h)})
aud = pd.DataFrame(audit_rows)
ref = ptbxl_db[["age", "sex", "scp_codes"]].copy()
ref["superclasses"] = ref.scp_codes.apply(ptbxl_superclasses)
aud = aud.merge(ref, left_on="ecg_id", right_index=True, how="inner")
demo_ok = (aud.ch_sex == aud.sex) & ((aud.ch_age - aud.age).abs() <= 1)
comparable = aud.ch_age.notna() & (aud.age <= 89) & (aud.ch_sex >= 0)
id_alignment = float(demo_ok[comparable].mean()) if comparable.any() else float("nan")
log(f"Mapping audit: {len(aud)} Challenge PTB-XL headers joined; ID alignment by age/sex = {id_alignment:.3f}")
MAPPING_AUDIT_VALID = bool(np.isfinite(id_alignment) and id_alignment >= 0.95)
aud = aud[aud.superclasses.str.len() > 0]
mapping_rows = []
for rule in NORM_COMPATIBLE_ABBR:
    mapped = aud.dx.apply(lambda d: map_dx(d, rule))
    ok = mapped.apply(lambda t: t[0] is not None)
    Ym = np.stack(mapped[ok].apply(lambda t: t[0]).values) if ok.any() else np.zeros((0, len(CLASSES)))
    Yr = np.array([[int(c in s) for c in CLASSES] for s in aud.superclasses[ok]]) if ok.any() else np.zeros((0, len(CLASSES)))
    for j, cls in enumerate(CLASSES):
        yr, ym = Yr[:, j], Ym[:, j]
        kappa = cohen_kappa_score(yr, ym) if len(np.unique(np.r_[yr, ym])) > 1 else np.nan
        mapping_rows.append({"norm_rule": rule, "class": cls, "n_records": int(ok.sum()),
                             "indeterminate_rate": float(1 - ok.mean()), "prevalence_ptbxl": float(yr.mean()) if len(yr) else np.nan,
                             "prevalence_mapped": float(ym.mean()) if len(ym) else np.nan, "cohen_kappa": kappa,
                             "sensitivity": float(ym[yr == 1].mean()) if (yr == 1).any() else np.nan,
                             "specificity": float(1 - ym[yr == 0].mean()) if (yr == 0).any() else np.nan,
                             "id_alignment": id_alignment, "audit_valid": MAPPING_AUDIT_VALID})
mapping_audit = pd.DataFrame(mapping_rows)
save_table(mapping_audit, "T01b_label_mapping_audit")
print(mapping_audit.round(3).to_string(index=False))
KAPPA_BY_CLASS = mapping_audit[mapping_audit.norm_rule == "A_primary"].set_index("class").cohen_kappa.to_dict()
if not MAPPING_AUDIT_VALID:
    log("WARNING: Challenge/PTB-XL record IDs could not be aligned (age/sex agreement < 0.95); the mapping audit is not "
        "valid and every external result is labelled exploratory.")

In [ ]:
# ---------- Load, preprocess and cache signals ----------
PREP_TAG = hashlib.md5(json.dumps([TARGET_FS, WINDOW_S, HIGHPASS_HZ, CLIP_MV, MODE, CFG["ptbxl_per_fold"],
                                   CFG["ext_subdirs"], CFG["ext_max_records"]]).encode()).hexdigest()[:8]

def build_cache(name, items, loader):
    """items: list of ids; loader(id) -> (x, status, extra dict). Cached as .npy + .csv keyed by PREP_TAG."""
    xp, mp = CACHE / f"{name}_{PREP_TAG}.npy", CACHE / f"{name}_{PREP_TAG}_meta.csv"
    if xp.exists() and mp.exists():
        meta = pd.read_csv(mp)
        t0 = time.time()
        # read the whole array into RAM in one sequential pass (memory-mapping a file on Google Drive makes every
        # later access a slow network read)
        local = WORK / "cache_local" / xp.name
        local.parent.mkdir(parents=True, exist_ok=True)
        if not local.exists() or local.stat().st_size != xp.stat().st_size:
            shutil.copyfile(xp, local)
        X = np.load(local)
        if len(meta) == X.shape[0]:
            log(f"cache hit {xp.name}: {X.shape} (loaded into memory in {time.time()-t0:.0f}s)")
            return X, meta
    Xs, rows, t0 = [], [], time.time()
    for i, it in enumerate(items, 1):
        try:
            x, status, extra = loader(it)
        except (HeaderParseError, ValueError, OSError, KeyError) as e:
            x, status, extra = None, f"read_error:{type(e).__name__}", {}
        rows.append({"id": it, "status": status, **extra})
        if x is not None:
            Xs.append(x)
        if i % 5000 == 0:
            print(f"  {name}: {i}/{len(items)} ({time.time()-t0:.0f}s)")
    meta = pd.DataFrame(rows)
    kept = meta[meta.status.str.startswith("ok")].reset_index(drop=True)
    X = np.stack(Xs).astype(np.float32) if Xs else np.zeros((0, N_TARGET, 12), np.float32)
    assert len(kept) == X.shape[0]
    save_table(meta.status.value_counts().rename_axis("status").reset_index(name="n"), f"S02_signal_status_{name}")
    np.save(xp, X); kept.to_csv(mp, index=False)
    log(f"built {name}: kept {X.shape[0]}/{len(items)}  ({time.time()-t0:.0f}s)")
    return X, kept

def load_ptbxl(ecg_id):
    sig, fs, h = read_signal(PTBXL_DIR / (ptbxl_sel.loc[ecg_id, "filename_hr"] + ".hea"))
    x, st = preprocess(sig, fs)
    return x, st, {}

X_src, meta_src = build_cache("ptbxl", ptbxl_sel.index.tolist(), load_ptbxl)
meta_src = meta_src.rename(columns={"id": "ecg_id"}).merge(
    ptbxl_sel[["patient_id", "age", "sex", "strat_fold", "superclasses"]].reset_index(), on="ecg_id", how="left")
meta_src["age"] = meta_src.age.where(meta_src.age <= 89, 90.0)   # PTB-XL encodes >89 years as 300
Y_src = np.array([[int(c in ast.literal_eval(s) if isinstance(s, str) else c in s) for c in CLASSES]
                  for s in meta_src.superclasses], dtype=np.int8)

EXT = {}
for key, folder in EXTERNAL_SITES.items():
    def load_ext(rec, _key=key):
        sig, fs, h = read_signal(CH_DIR / (rec + ".hea"))
        y, mstat = map_dx(header_dx(h), "A_primary")
        age, sex = header_demo(h)
        if y is None:
            return None, "label_indeterminate", {}
        x, st = preprocess(sig, fs)
        return x, st, {"age": age, "sex": sex, "fs_orig": fs, **{f"y_{c}": int(v) for c, v in zip(CLASSES, y)}}
    Xe, me = build_cache(f"ext_{key}", EXT_RECORDS[key], load_ext)
    EXT[key] = {"X": Xe, "meta": me, "Y": me[[f"y_{c}" for c in CLASSES]].to_numpy(np.int8)}

# Data quality / provenance report
qrows = [{"dataset": "PTB-XL", "records": len(meta_src), "patients": meta_src.patient_id.nunique(),
          "female_frac": float((meta_src.sex == 1).mean()), "age_median": float(meta_src.age.median()),
          **{f"prev_{c}": float(Y_src[:, j].mean()) for j, c in enumerate(CLASSES)}}]
for key, d in EXT.items():
    m = d["meta"]
    qrows.append({"dataset": key, "records": len(m), "patients": np.nan, "female_frac": float((m.sex == 1).mean()),
                  "age_median": float(m.age.median()), **{f"prev_{c}": float(d["Y"][:, j].mean()) for j, c in enumerate(CLASSES)}})
quality = pd.DataFrame(qrows)
save_table(quality, "T01_dataset_summary")
print(quality.round(3).to_string(index=False))
for name, X in [("PTB-XL", X_src)] + [(k, d["X"]) for k, d in EXT.items()]:
    assert X.ndim == 3 and X.shape[1:] == (N_TARGET, 12), f"{name}: bad shape {X.shape}"
    assert np.isfinite(X).all(), f"{name}: non-finite values after preprocessing"
    assert X.shape[0] >= (20 if MODE == "SMOKE_TEST" else 1000), f"{name}: too few records ({X.shape[0]})"
assert not meta_src.ecg_id.duplicated().any(), "duplicate PTB-XL ecg_id"
for key, d in EXT.items():
    assert not d["meta"].id.duplicated().any(), f"duplicate records in {key}"

hosts = {re.sub(r"^https?://([^/]+)/.*$", r"\1", u) for u in [PTBXL_BASE_URL, CHALLENGE_BASE_URL]}
DATA_PROVENANCE = {**PROVENANCE, "hosts": sorted(hosts), "prep_tag": PREP_TAG, "mapping_id_alignment": id_alignment,
                   "mapping_audit_valid": MAPPING_AUDIT_VALID, "n_ptbxl": int(len(meta_src)),
                   "n_external": {k: int(len(d["meta"])) for k, d in EXT.items()},
                   "preprocessing": f"resample to {TARGET_FS} Hz (polyphase, anti-aliased) from 500-Hz sources, first {WINDOW_S} s, "
                                    f"{HIGHPASS_HZ}-Hz 2nd-order zero-phase Butterworth high-pass, clip +/-{CLIP_MV} mV, "
                                    "per-lead z-score with training-set statistics"}
DATA_VERIFIED = hosts == {"physionet.org"} and all(
    v.get("files_without_checksum", 0) == 0 for v in PROVENANCE["sources"].values() if isinstance(v, dict))
DATA_PROVENANCE["verified_official_source"] = DATA_VERIFIED
save_json(DATA_PROVENANCE, "data_audit", "DATA_PROVENANCE")
print("DATA_PROVENANCE:", json.dumps({k: DATA_PROVENANCE[k] for k in ["hosts", "n_ptbxl", "n_external", "verified_official_source"]}))

In [ ]:
# ---------- Splits (patient-disjoint) and leakage assertions ----------
fold = meta_src.strat_fold.to_numpy()
idx_train = np.where(fold <= 8)[0]
idx_test = np.where(fold == 10)[0]
f9 = np.where(fold == 9)[0]
pats9 = np.array(sorted(meta_src.patient_id.iloc[f9].unique()))
np.random.default_rng(GLOBAL_SEED).shuffle(pats9)
tune_pats = set(pats9[: len(pats9) // 2])
idx_tune = np.array([i for i in f9 if meta_src.patient_id.iloc[i] in tune_pats])
idx_calib = np.array([i for i in f9 if meta_src.patient_id.iloc[i] not in tune_pats])
SPLITS = {"train": idx_train, "tune": idx_tune, "calib": idx_calib, "test": idx_test}
pat_sets = {k: set(meta_src.patient_id.iloc[v]) for k, v in SPLITS.items()}
for a in SPLITS:
    for b in SPLITS:
        if a < b:
            assert not (pat_sets[a] & pat_sets[b]), f"patient leakage between {a} and {b}"
            assert not (set(SPLITS[a]) & set(SPLITS[b])), f"record overlap between {a} and {b}"
assert sum(len(v) for v in SPLITS.values()) == len(meta_src)
split_table = pd.DataFrame([{"split": k, "role": r, "folds": f, "records": len(v), "patients": len(pat_sets[k]),
                             **{f"pos_{c}": int(Y_src[v, j].sum()) for j, c in enumerate(CLASSES)}}
                            for (k, v), r, f in zip(SPLITS.items(),
                                                    ["model fitting", "early stopping, temperature, F1 thresholds, HPO",
                                                     "conformal calibration only", "untouched test"],
                                                    ["1-8", "9 (half of patients)", "9 (other half)", "10"])])
save_table(split_table, "T00_split_table")
print(split_table.to_string(index=False))
print("Leakage assertions passed: train/tune/calib/test are patient- and record-disjoint.")

# Per-lead normalisation fitted on training records only
MU = X_src[idx_train].mean(axis=(0, 1)).astype(np.float32)
SD = X_src[idx_train].std(axis=(0, 1)).astype(np.float32) + 1e-6
save_json({"lead_mean": MU.tolist(), "lead_sd": SD.tolist(), "fitted_on": "train (folds 1-8)"}, "config", "normalisation")

def norm(X):
    return ((X - MU) / SD).astype(np.float32)

In [ ]:
# ---------- Hand-crafted features for the classical baselines ----------
BANDS = [(0.5, 4), (4, 15), (15, 40)]

def features(X):
    """184 features: 15 per lead (moments, percentiles, band-power fractions) + 4 rhythm features from lead II."""
    Xd = X.astype(np.float64)
    q = np.percentile(Xd, [5, 25, 50, 75, 95], axis=1)                      # (5, N, 12)
    f, P = sps.welch(Xd, fs=TARGET_FS, nperseg=256, axis=1)                  # P: (N, F, 12)
    tot = P[:, (f >= 0.5) & (f < 40)].sum(axis=1) + 1e-12
    bp = [P[:, (f >= lo) & (f < hi)].sum(axis=1) / tot for lo, hi in BANDS]
    per_lead = [Xd.mean(1), Xd.std(1), Xd.min(1), Xd.max(1), *q, sstats.skew(Xd, axis=1), sstats.kurtosis(Xd, axis=1),
                np.abs(np.diff(Xd, axis=1)).mean(1), *bp]
    per_lead = np.concatenate(per_lead, axis=1)                               # (N, 15*12)
    rr = np.zeros((X.shape[0], 4))
    lead2 = Xd[:, :, 1]
    for i in range(X.shape[0]):
        env = np.abs(np.gradient(lead2[i]))
        pk, _ = sps.find_peaks(env, distance=int(0.3 * TARGET_FS), height=np.percentile(env, 95) * 0.5)
        if len(pk) >= 3:
            ivals = np.diff(pk) / TARGET_FS
            rr[i] = [60 / ivals.mean(), ivals.std(), np.sqrt(np.mean(np.diff(ivals) ** 2)) if len(ivals) > 1 else 0, len(pk)]
        else:
            rr[i] = [0, 0, 0, len(pk)]
    out = np.concatenate([per_lead, rr], axis=1).astype(np.float32)
    return np.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0)

def cached_features(name, X):
    p = CACHE / f"feat_{name}_{PREP_TAG}.npy"
    if p.exists():
        F_ = np.load(p)
        if F_.shape[0] == X.shape[0]:
            return F_
    # batches of 2,000 ECGs keep peak RAM small (the full array in float64 would need several GB)
    F_ = np.concatenate([features(X[i:i + 2000]) for i in range(0, len(X), 2000)], axis=0)
    np.save(p, F_)
    return F_

t0 = time.time()
FEAT_SRC = cached_features("ptbxl", X_src)
FEAT_EXT = {k: cached_features(f"ext_{k}", d["X"]) for k, d in EXT.items()}
log(f"features: {FEAT_SRC.shape[1]} per record ({time.time()-t0:.0f}s)")
assert FEAT_SRC.shape[1] == 184

EVAL_SETS = ["tune", "calib", "test"] + [f"ext_{k}" for k in EXT]

def logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def macro_auroc(Y, P):
    vals = [roc_auc_score(Y[:, j], P[:, j]) for j in range(Y.shape[1]) if 0 < Y[:, j].sum() < len(Y)]
    return float(np.mean(vals)) if vals else float("nan")

def pred_path(model, seed, split):
    return SUB["predictions"] / f"{model}_seed{seed}_{split}.npz"

def save_preds(model, seed, logits_by_split, emb_by_split=None):
    for split, z in logits_by_split.items():
        payload = {"logits": z.astype(np.float32)}
        if emb_by_split is not None and split in emb_by_split:
            payload["emb"] = emb_by_split[split].astype(np.float16)
        np.savez_compressed(pred_path(model, seed, split), **payload)

def have_preds(model, seed):
    return all(pred_path(model, seed, s).exists() for s in EVAL_SETS)

def load_logits(model, seed, split):
    return np.load(pred_path(model, seed, split))["logits"]

def split_features(split):
    return FEAT_EXT[split[4:]] if split.startswith("ext_") else FEAT_SRC[SPLITS[split]]

def split_X(split):
    return EXT[split[4:]]["X"] if split.startswith("ext_") else X_src[SPLITS[split]]

def split_Y(split):
    return EXT[split[4:]]["Y"] if split.startswith("ext_") else Y_src[SPLITS[split]]

In [ ]:
# ---------- Baselines: prevalence, logistic regression, gradient boosting (validation-only tuning) ----------
Ftr, Ytr = FEAT_SRC[idx_train], Y_src[idx_train]
prev = Ytr.mean(0)
if not have_preds("prevalence", 0):
    save_preds("prevalence", 0, {s: np.tile(logit(prev), (len(split_Y(s)), 1)) for s in EVAL_SETS})

HPO_LOG = []
if not have_preds("logreg", 0):
    best = None
    for C in [0.01, 0.1, 1.0]:
        models = [make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=3000)).fit(Ftr, Ytr[:, j])
                  for j in range(len(CLASSES))]
        P = np.stack([m.predict_proba(split_features("tune"))[:, 1] for m in models], 1)
        auc = macro_auroc(Y_src[idx_tune], P)
        HPO_LOG.append({"model": "logreg", "param": f"C={C}", "tune_macro_auroc": auc})
        if best is None or auc > best[0]:
            best = (auc, C, models)
    _, C, models = best
    save_preds("logreg", 0, {s: logit(np.stack([m.predict_proba(split_features(s))[:, 1] for m in models], 1))
                             for s in EVAL_SETS})
    log(f"logreg: selected C={C} on tune split")

t0 = time.time()
hgb_leaves = None
for seed in CFG["hgb_seeds"]:
    if have_preds("hgb", seed):
        continue
    if hgb_leaves is None:
        best = None
        for leaves in [15, 31]:
            ms = [HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, max_leaf_nodes=leaves,
                                                 early_stopping=True, validation_fraction=0.1, random_state=seed)
                  .fit(Ftr, Ytr[:, j]) for j in range(len(CLASSES))]
            auc = macro_auroc(Y_src[idx_tune], np.stack([m.predict_proba(split_features("tune"))[:, 1] for m in ms], 1))
            HPO_LOG.append({"model": "hgb", "param": f"max_leaf_nodes={leaves}", "tune_macro_auroc": auc})
            if best is None or auc > best[0]:
                best = (auc, leaves, ms)
        hgb_leaves, ms = best[1], best[2]
    else:
        ms = [HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, max_leaf_nodes=hgb_leaves,
                                             early_stopping=True, validation_fraction=0.1, random_state=seed)
              .fit(Ftr, Ytr[:, j]) for j in range(len(CLASSES))]
    save_preds("hgb", seed, {s: logit(np.stack([m.predict_proba(split_features(s))[:, 1] for m in ms], 1)) for s in EVAL_SETS})
log(f"classical baselines done ({time.time()-t0:.0f}s)")
if HPO_LOG:
    save_table(pd.DataFrame(HPO_LOG), "S03_hpo_classical")

## Proposed backbone and deep-learning baseline
ResNet1d (proposed backbone): stem conv (k=15, stride 2) + max-pool, four residual stages (64/128/192/256 channels, two basic
blocks each, k=9), concatenated global average and max pooling (512-d embedding), dropout 0.3, linear head.
FCN (baseline; Wang et al. 2017): three conv blocks (128/256/128), global average pooling.
Both: AdamW (lr 1e-3, wd 1e-2), one-cycle schedule, binary cross-entropy, mixed precision on GPU, light augmentation
(amplitude scaling 0.9-1.1, Gaussian noise sd 0.02). Early stopping on tuning-split macro-AUROC only. Identical splits for all models.

In [ ]:
class BasicBlock1d(nn.Module):
    def __init__(self, cin, cout, stride, k=9):
        super().__init__()
        self.c1 = nn.Conv1d(cin, cout, k, stride, k // 2, bias=False); self.b1 = nn.BatchNorm1d(cout)
        self.c2 = nn.Conv1d(cout, cout, k, 1, k // 2, bias=False); self.b2 = nn.BatchNorm1d(cout)
        self.short = (nn.Sequential(nn.Conv1d(cin, cout, 1, stride, bias=False), nn.BatchNorm1d(cout))
                      if stride != 1 or cin != cout else nn.Identity())

    def forward(self, x):
        out = F.relu(self.b1(self.c1(x)))
        return F.relu(self.b2(self.c2(out)) + self.short(x))

class ResNet1d(nn.Module):
    def __init__(self, n_out=5, widths=(64, 128, 192, 256)):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv1d(12, widths[0], 15, 2, 7, bias=False), nn.BatchNorm1d(widths[0]), nn.ReLU(),
                                  nn.MaxPool1d(3, 2, 1))
        layers, cin = [], widths[0]
        for i, w in enumerate(widths):
            layers += [BasicBlock1d(cin, w, 1 if i == 0 else 2), BasicBlock1d(w, w, 1)]
            cin = w
        self.body = nn.Sequential(*layers)
        self.drop = nn.Dropout(0.3)
        self.head = nn.Linear(2 * widths[-1], n_out)
        self.emb_dim = 2 * widths[-1]

    def embed(self, x):
        h = self.body(self.stem(x))
        return torch.cat([h.mean(-1), h.amax(-1)], 1)

    def forward(self, x):
        return self.head(self.drop(self.embed(x)))

class FCN(nn.Module):
    def __init__(self, n_out=5):
        super().__init__()
        def blk(ci, co, k):
            return nn.Sequential(nn.Conv1d(ci, co, k, padding="same"), nn.BatchNorm1d(co), nn.ReLU())
        self.body = nn.Sequential(blk(12, 128, 8), blk(128, 256, 5), blk(256, 128, 3))
        self.head = nn.Linear(128, n_out)
        self.emb_dim = 128

    def embed(self, x):
        return self.body(x).mean(-1)

    def forward(self, x):
        return self.head(self.embed(x))

ARCHS = {"resnet1d": ResNet1d, "fcn": FCN}
for a, cls_ in ARCHS.items():
    _m = cls_(); _o = _m(torch.zeros(2, 12, N_TARGET))
    assert _o.shape == (2, len(CLASSES)), f"{a} output shape {_o.shape}"
    print(f"{a}: {sum(p.numel() for p in _m.parameters())/1e6:.2f} M parameters")
del _m, _o

AMP = DEVICE.type == "cuda"

@torch.no_grad()
def predict(model, X, bs=512, with_emb=False):
    model.eval()
    zs, es = [], []
    for i in range(0, len(X), bs):
        xb = torch.from_numpy(norm(X[i:i + bs])).permute(0, 2, 1).to(DEVICE)
        with torch.autocast(device_type=DEVICE.type, enabled=AMP):
            e = model.embed(xb)
            z = model.head(e)
        zs.append(z.float().cpu().numpy())
        if with_emb:
            es.append(e.float().cpu().numpy())
    return np.concatenate(zs), (np.concatenate(es) if with_emb else None)

def train_one(arch, seed):
    ck_dir = SUB["checkpoints"]; ck = ck_dir / f"{arch}_seed{seed}_last.pt"; best_p = SUB["models"] / f"{arch}_seed{seed}_best.pt"
    set_seed(seed)
    model = ARCHS[arch]().to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)
    bs, E = CFG["batch_size"], CFG["epochs"]
    steps = math.ceil(len(idx_train) / bs)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=1e-3, epochs=E, steps_per_epoch=steps)
    scaler = (torch.amp.GradScaler("cuda", enabled=AMP) if hasattr(torch.amp, "GradScaler")
              else torch.cuda.amp.GradScaler(enabled=AMP))
    state = {"epoch": 0, "best": -1.0, "best_epoch": -1, "bad": 0, "history": []}
    if ck.exists():
        c = torch.load(ck, map_location=DEVICE, weights_only=False)
        model.load_state_dict(c["model"]); opt.load_state_dict(c["opt"]); sched.load_state_dict(c["sched"])
        scaler.load_state_dict(c["scaler"]); state = c["state"]
        log(f"resumed {arch} seed {seed} at epoch {state['epoch']}")
    Xtr, Ytr_ = X_src[idx_train], Y_src[idx_train].astype(np.float32)
    rng = np.random.default_rng(seed + 1000 * (state["epoch"] + 1))
    while state["epoch"] < E and state["bad"] < CFG["patience"]:
        model.train(); t0 = time.time(); losses = []
        perm = rng.permutation(len(Xtr))
        for i in range(0, len(perm), bs):
            b = perm[i:i + bs]
            xb = norm(Xtr[b])
            xb = xb * rng.uniform(0.9, 1.1, (len(b), 1, 1)).astype(np.float32) + rng.normal(0, 0.02, xb.shape).astype(np.float32)
            xb = torch.from_numpy(xb).permute(0, 2, 1).to(DEVICE)
            yb = torch.from_numpy(Ytr_[b]).to(DEVICE)
            with torch.autocast(device_type=DEVICE.type, enabled=AMP):
                loss = F.binary_cross_entropy_with_logits(model(xb).float(), yb)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            losses.append(loss.item())
        z_tune, _ = predict(model, X_src[idx_tune])
        auc = macro_auroc(Y_src[idx_tune], sigmoid(z_tune))
        state["epoch"] += 1
        state["history"].append({"epoch": state["epoch"], "train_loss": float(np.mean(losses)), "tune_macro_auroc": auc,
                                 "epoch_seconds": time.time() - t0})
        if auc > state["best"] + 1e-4:
            state.update(best=auc, best_epoch=state["epoch"], bad=0)
            torch.save(model.state_dict(), best_p)
        else:
            state["bad"] += 1
        torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict(),
                    "scaler": scaler.state_dict(), "state": state}, ck)
        print(f"  {arch} s{seed} ep{state['epoch']:02d} loss {np.mean(losses):.4f} tune-AUROC {auc:.4f} "
              f"({time.time()-t0:.0f}s){' *' if state['best_epoch']==state['epoch'] else ''}")
    model.load_state_dict(torch.load(best_p, map_location=DEVICE))
    return model, state

TRAIN_LOG = []
t_all = time.time()
for arch in CFG["archs"]:
    for seed in CFG["seeds"]:
        hist_p = SUB["logs"] / f"history_{arch}_seed{seed}.csv"
        if have_preds(arch, seed) and (SUB["models"] / f"{arch}_seed{seed}_best.pt").exists() and hist_p.exists():
            TRAIN_LOG.append(pd.read_csv(hist_p).assign(arch=arch, seed=seed))
            continue
        model, st = train_one(arch, seed)
        want_emb = arch == "resnet1d"
        logits, embs = {}, {}
        for s in EVAL_SETS:
            z, e = predict(model, split_X(s), with_emb=want_emb)
            logits[s] = z
            if want_emb:
                embs[s] = e
        save_preds(arch, seed, logits, embs if want_emb else None)
        h = pd.DataFrame(st["history"]); h.to_csv(hist_p, index=False)
        TRAIN_LOG.append(h.assign(arch=arch, seed=seed))
        log(f"{arch} seed {seed}: best tune macro-AUROC {st['best']:.4f} at epoch {st['best_epoch']}")
        del model
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
TRAIN_HIST = pd.concat(TRAIN_LOG, ignore_index=True)
save_table(TRAIN_HIST, "S04_training_history")
log(f"deep models done ({(time.time()-t_all)/60:.1f} min)")

## Calibration and in-domain evaluation (PTB-XL fold 10, untouched until here)
Per-label temperature scaling is fitted on the **tune** split for every model. Decision thresholds for F1 and balanced accuracy
are chosen per label on the tune split (maximum F1). Test-set results are computed once with frozen parameters.
Uncertainty: patient-cluster bootstrap 95% CIs on fold 10; seed mean +/- SD for single deep models.

In [ ]:
def fit_temperature(z, y):
    """Per-label temperature minimising binary NLL on the tune split."""
    T = np.ones(z.shape[1])
    for j in range(z.shape[1]):
        def nll(logT):
            p = np.clip(sigmoid(z[:, j] / np.exp(logT)), 1e-7, 1 - 1e-7)
            return -np.mean(y[:, j] * np.log(p) + (1 - y[:, j]) * np.log(1 - p))
        T[j] = float(np.exp(minimize_scalar(nll, bounds=(np.log(0.05), np.log(20)), method="bounded").x))
    return T

def ece_classwise(Y, P, n_bins=15):
    """Mean over labels of equal-mass-binned ECE."""
    out = []
    for j in range(Y.shape[1]):
        p, y = P[:, j], Y[:, j]
        order = np.argsort(p)
        bins = np.array_split(order, n_bins)
        out.append(sum(len(b) / len(p) * abs(p[b].mean() - y[b].mean()) for b in bins if len(b)))
    return float(np.mean(out)), out

def best_f1_thresholds(Y, P):
    th = np.zeros(Y.shape[1])
    grid = np.linspace(0.02, 0.98, 97)
    for j in range(Y.shape[1]):
        f = [f1_score(Y[:, j], P[:, j] >= t, zero_division=0) for t in grid]
        th[j] = grid[int(np.argmax(f))]
    return th

def metrics(Y, P, th, labels=None):
    labels = labels if labels is not None else list(range(Y.shape[1]))
    ok = [j for j in labels if 0 < Y[:, j].sum() < len(Y)]
    res = {"macro_auroc": np.mean([roc_auc_score(Y[:, j], P[:, j]) for j in ok]) if ok else np.nan,
           "macro_auprc": np.mean([average_precision_score(Y[:, j], P[:, j]) for j in ok]) if ok else np.nan,
           "macro_f1": np.mean([f1_score(Y[:, j], P[:, j] >= th[j], zero_division=0) for j in ok]) if ok else np.nan,
           "macro_bal_acc": np.mean([balanced_accuracy_score(Y[:, j], P[:, j] >= th[j]) for j in ok]) if ok else np.nan,
           "brier": float(np.mean((P[:, ok] - Y[:, ok]) ** 2)) if ok else np.nan}
    res["ece"] = ece_classwise(Y[:, ok], P[:, ok])[0] if ok else np.nan
    return res

# Assemble calibrated predictions for every model variant
MODEL_SEEDS = {"prevalence": [0], "logreg": [0], "hgb": CFG["hgb_seeds"], **{a: CFG["seeds"] for a in CFG["archs"]}}
PRED = {}   # PRED[(variant)][split] = dict(z_raw, p_raw, p_cal)
def register(name, z_by_split):
    T = fit_temperature(z_by_split["tune"], Y_src[idx_tune])
    PRED[name] = {"T": T}
    for s, z in z_by_split.items():
        PRED[name][s] = {"z": z, "p_raw": sigmoid(z), "p_cal": sigmoid(z / T)}

for m, seeds in MODEL_SEEDS.items():
    zs = {s: [load_logits(m, sd, s) for sd in seeds] for s in EVAL_SETS}
    register(f"{m}_ens" if len(seeds) > 1 else m, {s: np.mean(v, 0) for s, v in zs.items()})
    if m in CFG["archs"]:
        for k, sd in enumerate(seeds):
            register(f"{m}_seed{sd}", {s: v[k] for s, v in zs.items()})
PROPOSED = "resnet1d_ens" if len(CFG["seeds"]) > 1 else "resnet1d"
BASELINES = ["prevalence", "logreg", "hgb_ens" if len(CFG["hgb_seeds"]) > 1 else "hgb",
             "fcn_ens" if len(CFG["seeds"]) > 1 else "fcn"]
if "resnet1d" not in PRED:
    PRED["resnet1d"] = PRED[f"resnet1d_seed{CFG['seeds'][0]}"]
temps = pd.DataFrame({k: v["T"] for k, v in PRED.items()}, index=CLASSES).T
save_table(temps.reset_index().rename(columns={"index": "model"}), "S05_temperatures")

TH = {k: best_f1_thresholds(Y_src[idx_tune], v["tune"]["p_cal"]) for k, v in PRED.items()}
Yte = Y_src[idx_test]
pat_test = meta_src.patient_id.to_numpy()[idx_test]
_, pat_codes = np.unique(pat_test, return_inverse=True)
groups = [np.where(pat_codes == g)[0] for g in range(pat_codes.max() + 1)]

def cluster_boot_indices(B, seed=GLOBAL_SEED):
    rng = np.random.default_rng(seed)
    for _ in range(B):
        pick = rng.integers(0, len(groups), len(groups))
        yield np.concatenate([groups[g] for g in pick])

BOOT = list(cluster_boot_indices(CFG["n_boot"]))
rows = []
main_models = BASELINES + [PROPOSED]
for name in main_models:
    P = PRED[name]["test"]["p_cal"]
    point = metrics(Yte, P, TH[name])
    bs = {k: [] for k in ["macro_auroc", "macro_auprc", "macro_f1"]}
    if name != "prevalence":
        for ix in BOOT:
            if Yte[ix].sum(0).min() == 0:
                continue
            bs["macro_auroc"].append(macro_auroc(Yte[ix], P[ix]))
            bs["macro_auprc"].append(np.mean([average_precision_score(Yte[ix, j], P[ix, j]) for j in range(len(CLASSES))]))
            bs["macro_f1"].append(np.mean([f1_score(Yte[ix, j], P[ix, j] >= TH[name][j], zero_division=0) for j in range(len(CLASSES))]))
    row = {"model": name, **{k: point[k] for k in ["macro_auroc", "macro_auprc", "macro_f1", "macro_bal_acc"]}}
    for k, v in bs.items():
        row[f"{k}_ci_low"], row[f"{k}_ci_high"] = (np.percentile(v, [2.5, 97.5]) if v else (np.nan, np.nan))
    for j, c in enumerate(CLASSES):
        row[f"auroc_{c}"] = roc_auc_score(Yte[:, j], P[:, j]) if 0 < Yte[:, j].sum() < len(Yte) else np.nan
    rows.append(row)
# seed-level mean +/- SD for single deep models
for a in CFG["archs"]:
    vals = pd.DataFrame([metrics(Yte, PRED[f"{a}_seed{sd}"]["test"]["p_cal"], TH[f"{a}_seed{sd}"]) for sd in CFG["seeds"]])
    rows.append({"model": f"{a}_single_seed_mean", **vals.mean().to_dict(),
                 **{f"{k}_sd": v for k, v in vals.std(ddof=1).to_dict().items()}, "n_seeds": len(CFG["seeds"])})
T02 = pd.DataFrame(rows)
save_table(T02, "T02_discrimination_indomain")
print(T02[["model", "macro_auroc", "macro_auroc_ci_low", "macro_auroc_ci_high", "macro_auprc", "macro_f1"]].round(4).to_string(index=False))

cal_rows = []
for name in main_models:
    for kind in ["p_raw", "p_cal"]:
        P = PRED[name]["test"][kind]
        e, per = ece_classwise(Yte, P)
        cal_rows.append({"model": name, "calibration": "none" if kind == "p_raw" else "temperature", "macro_ece": e,
                         "brier": float(np.mean((P - Yte) ** 2)), **{f"ece_{c}": v for c, v in zip(CLASSES, per)}})
T03 = pd.DataFrame(cal_rows)
save_table(T03, "T03_calibration_indomain")
print(T03[["model", "calibration", "macro_ece", "brier"]].round(4).to_string(index=False))

In [ ]:
# ---------- Paired statistics: proposed vs each baseline (patient-cluster bootstrap, Holm) ----------
def paired_delta(a, b):
    Pa, Pb = PRED[a]["test"]["p_cal"], PRED[b]["test"]["p_cal"]
    d = [macro_auroc(Yte[ix], Pa[ix]) - macro_auroc(Yte[ix], Pb[ix]) for ix in BOOT if Yte[ix].sum(0).min() > 0]
    d = np.array(d)
    p = min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean()))
    return macro_auroc(Yte, Pa) - macro_auroc(Yte, Pb), np.percentile(d, 2.5), np.percentile(d, 97.5), max(p, 1 / len(d))

test_rows = []
for b in [m for m in BASELINES if m != "prevalence"]:
    dlt, lo, hi, p = paired_delta(PROPOSED, b)
    test_rows.append({"comparison": f"{PROPOSED} vs {b}", "delta_macro_auroc": dlt, "ci_low": lo, "ci_high": hi, "p_boot": p})
# seed-level paired Wilcoxon (low power with <=5 seeds; reported for completeness)
if len(CFG["seeds"]) >= 3 and "fcn" in CFG["archs"]:
    ra = [macro_auroc(Yte, PRED[f"resnet1d_seed{s}"]["test"]["p_cal"]) for s in CFG["seeds"]]
    fa = [macro_auroc(Yte, PRED[f"fcn_seed{s}"]["test"]["p_cal"]) for s in CFG["seeds"]]
    w = sstats.wilcoxon(ra, fa)
    diff = np.array(ra) - np.array(fa)
    test_rows.append({"comparison": "resnet1d vs fcn (per-seed Wilcoxon)", "delta_macro_auroc": float(diff.mean()),
                      "ci_low": np.nan, "ci_high": np.nan, "p_boot": float(w.pvalue),
                      "effect_size_dz": float(diff.mean() / diff.std(ddof=1)) if diff.std(ddof=1) > 0 else np.nan})
T04 = pd.DataFrame(test_rows)
order = np.argsort(T04.p_boot.values)
adj = np.empty(len(T04)); running = 0
for rank, i in enumerate(order):
    running = max(running, min(1, (len(T04) - rank) * T04.p_boot.values[i])); adj[i] = running
T04["p_holm"] = adj
save_table(T04, "T04_paired_tests")
print(T04.round(4).to_string(index=False))

## Label-conditional (Mondrian) conformal prediction, in-domain
For each superclass k the prediction set is a subset of {0, 1}. Non-conformity: s = 1 - p_k(y). Thresholds are the
ceil((n+1)(1 - alpha))-th smallest calibration score, computed **separately for calibration ECGs with y = 1 and y = 0** (Mondrian).
Guarantee (exchangeability): P(y in C | y = c) >= 1 - alpha for c in {0, 1} and every label. For positives this means a
diseased ECG is confidently labelled "absent" with probability <= alpha (rule-out safety). Sets {1} or {0} are automatic
calls; {0, 1} (uncertain) and the empty set are referred for review. The global variant pools both classes of a label.

In [ ]:
def cp_quantile(scores, alpha):
    n = len(scores)
    k = math.ceil((n + 1) * (1 - alpha))
    if n == 0 or k > n:
        return np.inf
    return float(np.sort(scores)[k - 1])

def mondrian_Q(P, Y, alpha):
    Q = np.full((P.shape[1], 2), np.inf)
    for j in range(P.shape[1]):
        Q[j, 1] = cp_quantile(1 - P[Y[:, j] == 1, j], alpha)
        Q[j, 0] = cp_quantile(P[Y[:, j] == 0, j], alpha)
    return Q

def global_Q(P, Y, alpha):
    Q = np.full((P.shape[1], 2), np.inf)
    for j in range(P.shape[1]):
        s = np.where(Y[:, j] == 1, 1 - P[:, j], P[:, j])
        Q[j, :] = cp_quantile(s, alpha)
    return Q

def cp_sets(P, q0, q1):
    """q0, q1 broadcastable to P. Returns boolean arrays inc0, inc1."""
    return P <= q0, (1 - P) <= q1

def cp_eval(Y, P, inc0, inc1, labels=None, alpha=ALPHA_PRIMARY):
    labels = labels if labels is not None else range(Y.shape[1])
    rows = []
    for j in labels:
        y = Y[:, j]
        size = inc0[:, j].astype(int) + inc1[:, j].astype(int)
        for c, inc in [(1, inc1[:, j]), (0, inc0[:, j])]:
            m = y == c
            n, k = int(m.sum()), int(inc[m].sum())
            ci = sstats.binomtest(k, n).proportion_ci(method="exact") if n > 0 else None
            rows.append({"class": CLASSES[j], "true_class": c, "n": n, "coverage": k / n if n else np.nan,
                         "cov_ci_low": ci.low if ci else np.nan, "cov_ci_high": ci.high if ci else np.nan,
                         "singleton_correct_rate": float(((size == 1) & inc)[m].mean()) if n else np.nan})
        rows[-1].update(mean_set_size=float(size.mean()), singleton_rate=float((size == 1).mean()),
                        uncertain_rate=float((size == 2).mean()), empty_rate=float((size == 0).mean()))
        rows[-2].update(mean_set_size=float(size.mean()), singleton_rate=float((size == 1).mean()),
                        uncertain_rate=float((size == 2).mean()), empty_rate=float((size == 0).mean()))
    return pd.DataFrame(rows)

def record_level(Y, inc0, inc1, labels=None):
    labels = list(labels if labels is not None else range(Y.shape[1]))
    single = (inc0[:, labels] ^ inc1[:, labels])
    auto = single.all(1)
    pred = inc1[:, labels] & ~inc0[:, labels]
    exact = (pred == Y[:, labels].astype(bool)).all(1)
    return {"auto_report_rate": float(auto.mean()), "referral_rate": float(1 - auto.mean()),
            "exact_match_auto": float(exact[auto].mean()) if auto.any() else np.nan,
            "exact_match_referred": float(exact[~auto].mean()) if (~auto).any() else np.nan}

Ycal = Y_src[idx_calib]
CP_ROWS, REC_ROWS, QSTORE = [], [], {}
for alpha in CFG["alphas"]:
    for name in [PROPOSED] + [m for m in BASELINES if m != "prevalence"]:
        Pc, Pt = PRED[name]["calib"]["p_cal"], PRED[name]["test"]["p_cal"]
        for variant, fn in [("mondrian", mondrian_Q), ("global", global_Q)]:
            Q = fn(Pc, Ycal, alpha)
            QSTORE[(name, variant, alpha)] = Q
            i0, i1 = cp_sets(Pt, Q[:, 0], Q[:, 1])
            df = cp_eval(Yte, Pt, i0, i1).assign(model=name, cp=variant, alpha=alpha)
            CP_ROWS.append(df)
            REC_ROWS.append({"model": name, "cp": variant, "alpha": alpha, **record_level(Yte, i0, i1)})
T05 = pd.concat(CP_ROWS, ignore_index=True)
T05b = pd.DataFrame(REC_ROWS)
save_table(T05, "T05_conformal_indomain")
save_table(T05b, "T05b_conformal_record_level")
print(T05[(T05.model == PROPOSED) & (T05.alpha == ALPHA_PRIMARY)][["cp", "class", "true_class", "n", "coverage", "cov_ci_low",
                                                                   "mean_set_size", "uncertain_rate"]].round(3).to_string(index=False))
save_json({f"{k[0]}|{k[1]}|{k[2]}": v.tolist() for k, v in QSTORE.items()}, "models", "conformal_thresholds_source")

In [ ]:
# ---------- Subgroup validity (sex, age bands) and subgroup discrimination ----------
AGE_BINS = [0, 40, 60, 75, 200]; AGE_LABELS = ["<40", "40-59", "60-74", ">=75"]
def subgroups(meta):
    sex = meta.sex.map({0: "male", 1: "female"}).fillna("unknown")
    age = pd.cut(meta.age, AGE_BINS, right=False, labels=AGE_LABELS).astype(str).replace("nan", "unknown")
    return {"sex": sex.to_numpy(), "age": age.to_numpy()}

SG_ROWS = []
Q = QSTORE[(PROPOSED, "mondrian", ALPHA_PRIMARY)]
Pt = PRED[PROPOSED]["test"]["p_cal"]
i0, i1 = cp_sets(Pt, Q[:, 0], Q[:, 1])
sg = subgroups(meta_src.iloc[idx_test])
for dim, vals in sg.items():
    for g in sorted(set(vals)):
        m = vals == g
        if g == "unknown" or m.sum() < 30:
            continue
        for j, c in enumerate(CLASSES):
            pos = m & (Yte[:, j] == 1)
            SG_ROWS.append({"dimension": dim, "group": g, "class": c, "n_records": int(m.sum()), "n_pos": int(pos.sum()),
                            "pos_coverage": float(i1[pos, j].mean()) if pos.sum() else np.nan,
                            "neg_coverage": float(i0[m & (Yte[:, j] == 0), j].mean()),
                            "auroc": roc_auc_score(Yte[m, j], Pt[m, j]) if 0 < Yte[m, j].sum() < m.sum() else np.nan})
T06 = pd.DataFrame(SG_ROWS)
save_table(T06, "T06_subgroup_coverage")
print(T06.round(3).to_string(index=False))

In [ ]:
# ---------- Ablations (one component changed at a time, all on PTB-XL fold 10, alpha = 0.10) ----------
def ablation_row(label, name, cal_kind, cp_variant):
    Pc, Pt = PRED[name]["calib"][cal_kind], PRED[name]["test"][cal_kind]
    Qa = (mondrian_Q if cp_variant == "mondrian" else global_Q)(Pc, Ycal, ALPHA_PRIMARY)
    a0, a1 = cp_sets(Pt, Qa[:, 0], Qa[:, 1])
    ev = cp_eval(Yte, Pt, a0, a1)
    pos = ev[ev.true_class == 1]
    return {"variant": label, "model": name, "calibration": cal_kind, "cp": cp_variant,
            "macro_auroc": macro_auroc(Yte, Pt), "macro_ece": ece_classwise(Yte, Pt)[0],
            "min_pos_coverage": float(pos.coverage.min()), "min_stratum_coverage": float(ev.coverage.min()),
            "mean_set_size": float(ev.drop_duplicates("class").mean_set_size.mean()),
            **record_level(Yte, a0, a1)}

single = f"resnet1d_seed{CFG['seeds'][0]}"
abl = [ablation_row("Full framework", PROPOSED, "p_cal", "mondrian"),
       ablation_row("- temperature scaling", PROPOSED, "p_raw", "mondrian"),
       ablation_row("- Mondrian (global CP)", PROPOSED, "p_cal", "global")]
if single != PROPOSED:
    abl.append(ablation_row("- ensemble (single seed)", single, "p_cal", "mondrian"))
fcn_name = "fcn_ens" if len(CFG["seeds"]) > 1 else "fcn"
if fcn_name in PRED:
    abl.append(ablation_row("ResNet1d -> FCN backbone", fcn_name, "p_cal", "mondrian"))
T07 = pd.DataFrame(abl)
save_table(T07, "T07_ablations")
print(T07.round(4).to_string(index=False))

## Post-hoc extensions (added after the first FULL_PAPER_RUN; reported as exploratory, not predeclared)
Motivated by two findings of the predeclared analysis: (i) one in-domain stratum fell just below the 0.88 gate, and
(ii) positive-class coverage was low in younger patients (G4). The primary gates and verdict above are **not** changed.
- **T16 finite-sample check.** Under exchangeability, split-conformal coverage given the calibration set is
  Beta(k, n+1-k) with k = ceil((n+1)(1-alpha)). Combined with binomial test sampling, this gives the exact distribution
  of observed test coverage for each stratum; one-sided p-values (Holm-adjusted) test for under-coverage.
- **T15 repeated calibration re-splits.** Fold 9 is re-split by patient 200 times into tune/calibration halves; the
  temperature is refitted on each tune half and Mondrian thresholds on each calibration half; coverage is measured on the
  same untouched fold 10. This shows how much coverage moves with the calibration draw alone.
- **T14 age-aware Mondrian conformal.** Thresholds stratified by label x true class x age band (<40, 40-59, 60-74, >=75);
  a band with fewer than 30 calibration cases for a stratum borrows the adjacent band (<40 with 40-59, >=75 with 60-74).

In [ ]:
N_RESPLITS = 200 if MODE == "FULL_PAPER_RUN" else (50 if MODE == "STANDARD_RUN" else 10)
AGE_MIN_STRATUM = 30
AGE_MERGE = {"<40": ["<40", "40-59"], "40-59": ["40-59", "<40"], "60-74": ["60-74", ">=75"], ">=75": [">=75", "60-74"]}

# ---------- T16: exact finite-sample check of in-domain coverage ----------
rng16 = np.random.default_rng(GLOBAL_SEED + 16)
t16 = []
for alpha in CFG["alphas"]:
    Q = QSTORE[(PROPOSED, "mondrian", alpha)]
    ev = T05[(T05.model == PROPOSED) & (T05.cp == "mondrian") & (T05.alpha == alpha)]
    for _, r in ev.iterrows():
        j = CLASSES.index(r["class"])
        n_cal = int((Ycal[:, j] == r.true_class).sum())
        k = math.ceil((n_cal + 1) * (1 - alpha))
        if k > n_cal or r.n == 0:
            continue
        C = rng16.beta(k, n_cal + 1 - k, 200_000)
        sim = rng16.binomial(int(r.n), C) / r.n
        t16.append({"alpha": alpha, "class": r["class"], "true_class": int(r.true_class), "n_calib": n_cal, "n_test": int(r.n),
                    "expected_coverage": k / (n_cal + 1), "sd_total": float(sim.std()), "observed": float(r.coverage),
                    "p_undercoverage": float((sim <= r.coverage).mean())})
T16 = pd.DataFrame(t16)
T16["p_holm"] = np.nan
for alpha, grp in T16.groupby("alpha"):
    order = grp.sort_values("p_undercoverage").index
    running = 0.0
    for rank, i in enumerate(order):
        running = max(running, min(1.0, (len(order) - rank) * T16.loc[i, "p_undercoverage"]))
        T16.loc[i, "p_holm"] = running
save_table(T16, "T16_finite_sample_coverage_check")
print(T16.round(4).to_string(index=False))

# ---------- T15: repeated patient-level re-splits of fold 9 ----------
f9_idx = np.concatenate([idx_tune, idx_calib])
z9 = np.vstack([PRED[PROPOSED]["tune"]["z"], PRED[PROPOSED]["calib"]["z"]])
y9 = Y_src[f9_idx]
p9 = meta_src.patient_id.to_numpy()[f9_idx]
upats = np.unique(p9)
z_test = PRED[PROPOSED]["test"]["z"]
rs_rows = []
t0 = time.time()
for rep in range(N_RESPLITS):
    rng = np.random.default_rng(50_000 + rep)
    perm = rng.permutation(upats)
    tune_set = set(perm[: len(perm) // 2])
    mt = np.array([p in tune_set for p in p9])
    T_rep = fit_temperature(z9[mt], y9[mt])
    pc, pt = sigmoid(z9[~mt] / T_rep), sigmoid(z_test / T_rep)
    for alpha in CFG["alphas"]:
        Qr = mondrian_Q(pc, y9[~mt], alpha)
        i0, i1 = cp_sets(pt, Qr[:, 0], Qr[:, 1])
        for j, c in enumerate(CLASSES):
            for cls, inc in ((1, i1[:, j]), (0, i0[:, j])):
                m = Yte[:, j] == cls
                rs_rows.append({"rep": rep, "alpha": alpha, "class": c, "true_class": cls, "coverage": float(inc[m].mean())})
RS = pd.DataFrame(rs_rows)
save_table(RS, "S07_calibration_resplits_raw")
orig = T05[(T05.model == PROPOSED) & (T05.cp == "mondrian")][["alpha", "class", "true_class", "coverage"]].rename(
    columns={"coverage": "original_split"})
T15 = (RS.groupby(["alpha", "class", "true_class"]).coverage
       .agg(mean="mean", sd="std", p05=lambda x: np.percentile(x, 5), p95=lambda x: np.percentile(x, 95)).reset_index())
T15 = T15.merge(orig, on=["alpha", "class", "true_class"], how="left")
T15["frac_repeats_ge_gate"] = [float((RS[(RS.alpha == r.alpha) & (RS["class"] == r["class"]) & (RS.true_class == r.true_class)]
                                      .coverage >= 1 - r.alpha - 0.02).mean()) for _, r in T15.iterrows()]
T15["original_percentile"] = [float((RS[(RS.alpha == r.alpha) & (RS["class"] == r["class"]) & (RS.true_class == r.true_class)]
                                     .coverage <= r.original_split).mean()) for _, r in T15.iterrows()]
save_table(T15, "T15_calibration_resplits_summary")
print(T15.round(3).to_string(index=False))
log(f"re-splits: {N_RESPLITS} repeats ({time.time()-t0:.0f}s)")

# ---------- T14: age-aware Mondrian conformal ----------
age_cal = subgroups(meta_src.iloc[idx_calib])["age"]
age_te = subgroups(meta_src.iloc[idx_test])["age"]
Pc_prop, Pt_prop = PRED[PROPOSED]["calib"]["p_cal"], PRED[PROPOSED]["test"]["p_cal"]

def age_aware_sets(alpha):
    i0 = np.zeros_like(Pt_prop, dtype=bool); i1 = np.zeros_like(Pt_prop, dtype=bool)
    fallback = []
    Qglob = mondrian_Q(Pc_prop, Ycal, alpha)
    for band in AGE_LABELS + ["unknown"]:
        mt = age_te == band
        if not mt.any():
            continue
        for j, c in enumerate(CLASSES):
            for cls in (0, 1):
                used = None
                for cand in (AGE_MERGE.get(band, [])[:1], AGE_MERGE.get(band, [])):
                    mc = np.isin(age_cal, cand) & (Ycal[:, j] == cls)
                    if cand and mc.sum() >= AGE_MIN_STRATUM:
                        used = cand
                        break
                if used is None:
                    q = Qglob[j, cls]; src = "class-level"
                else:
                    sc = (1 - Pc_prop[mc, j]) if cls == 1 else Pc_prop[mc, j]
                    q = cp_quantile(sc, alpha); src = "+".join(used)
                if cls == 1:
                    i1[mt, j] = (1 - Pt_prop[mt, j]) <= q
                else:
                    i0[mt, j] = Pt_prop[mt, j] <= q
                fallback.append({"alpha": alpha, "age_band": band, "class": c, "true_class": cls, "threshold_from": src})
    return i0, i1, pd.DataFrame(fallback)

t14, fb_all = [], []
for alpha in CFG["alphas"]:
    Qc = QSTORE[(PROPOSED, "mondrian", alpha)]
    c0, c1 = cp_sets(Pt_prop, Qc[:, 0], Qc[:, 1])
    a0, a1, fb = age_aware_sets(alpha)
    fb_all.append(fb)
    for variant, (s0, s1) in {"class_mondrian": (c0, c1), "age_aware_mondrian": (a0, a1)}.items():
        for band in AGE_LABELS:
            m = age_te == band
            for j, c in enumerate(CLASSES):
                for cls, inc in ((1, s1[:, j]), (0, s0[:, j])):
                    mm = m & (Yte[:, j] == cls)
                    if mm.sum() == 0:
                        continue
                    k_, n_ = int(inc[mm].sum()), int(mm.sum())
                    ci = sstats.binomtest(k_, n_).proportion_ci(method="exact")
                    t14.append({"alpha": alpha, "variant": variant, "age_band": band, "class": c, "true_class": cls,
                                "n": n_, "coverage": k_ / n_, "ci_low": ci.low, "ci_high": ci.high})
        rl = record_level(Yte, s0, s1)
        size = (s0.astype(int) + s1.astype(int)).mean()
        t14.append({"alpha": alpha, "variant": variant, "age_band": "ALL", "class": "record-level", "true_class": -1,
                    "n": len(Yte), "coverage": np.nan, "referral_rate": rl["referral_rate"], "mean_set_size": float(size)})
T14 = pd.DataFrame(t14)
save_table(T14, "T14_age_aware_conformal")
save_table(pd.concat(fb_all, ignore_index=True), "S08_age_aware_threshold_sources")
elig = T14[(T14.true_class == 1) & (T14.n >= 30) & (T14.alpha == ALPHA_PRIMARY)]
print(elig.pivot_table(index=["age_band", "class"], columns="variant", values="coverage").round(3).to_string())
print(T14[T14.age_band == "ALL"][["alpha", "variant", "referral_rate", "mean_set_size"]].round(3).to_string(index=False))

## External validation under cross-institution shift (Georgia, Chapman-Shaoxing)
Frozen source model and source temperatures. A class is evaluable at a site only if it has >= `min_pos_ext` positives and
negatives (predeclared); MI is not evaluable at either site. Each repeat splits a site at random into a labelled adaptation
budget A (first n of the permutation, nested across n), an unlabelled pool U (used only by weighted methods), and an evaluation
set E (identical for every method and budget within a repeat).

Site-adaptation methods (Mondrian thresholds per label and class):
- **M0 source**: thresholds from the PTB-XL calibration split (no adaptation).
- **M1 target**: thresholds from the labelled budget A only (valid by exchangeability within the site; infinite threshold,
  i.e. always include, when a stratum has fewer than ceil(1/alpha) - 1 cases).
- **M2 weighted**: covariate-shift-weighted source thresholds (Tibshirani et al. 2019) with a logistic domain classifier on the
  ResNet1d embedding (PCA-32); uses only unlabelled target ECGs.
- **M3 hybrid**: M1 where a stratum has >= `N_MIN_STRATUM_HYBRID` labelled target cases, M2 otherwise.
- **M4 pooled**: thresholds from the source calibration split plus A, unweighted.

In [ ]:
EXT_ROWS, EXT_CAL_ROWS = [], []
EVALUABLE = {}
emb_seed = CFG["seeds"][0]
for key, d in EXT.items():
    Ye = d["Y"]
    labels = [j for j, c in enumerate(CLASSES) if Ye[:, j].sum() >= CFG["min_pos_ext"] and (1 - Ye[:, j]).sum() >= CFG["min_pos_ext"]]
    EVALUABLE[key] = labels
    for name in main_models:
        Pe = PRED[name][f"ext_{key}"]["p_cal"]
        mt = metrics(Ye, Pe, TH[name], labels)
        row = {"site": key, "model": name, "n": len(Ye), "evaluable_classes": ",".join(CLASSES[j] for j in labels), **mt}
        for j in labels:
            row[f"auroc_{CLASSES[j]}"] = roc_auc_score(Ye[:, j], Pe[:, j])
        # bootstrap CI for macro AUROC (record-level; no patient IDs available)
        if name != "prevalence" and labels:
            rng = np.random.default_rng(GLOBAL_SEED)
            bs = []
            for _ in range(CFG["n_boot"]):
                ix = rng.integers(0, len(Ye), len(Ye))
                if Ye[np.ix_(ix, labels)].sum(0).min() > 0:
                    bs.append(np.mean([roc_auc_score(Ye[ix, j], Pe[ix, j]) for j in labels]))
            row["macro_auroc_ci_low"], row["macro_auroc_ci_high"] = np.percentile(bs, [2.5, 97.5])
        EXT_ROWS.append(row)
    # in-domain reference on the same evaluable classes
    EXT_ROWS.append({"site": f"PTB-XL test (classes of {key})", "model": PROPOSED, "n": len(Yte),
                     **metrics(Yte, PRED[PROPOSED]["test"]["p_cal"], TH[PROPOSED], labels)})
T08 = pd.DataFrame(EXT_ROWS)
save_table(T08, "T08_external_discrimination_calibration")
print(T08[["site", "model", "n", "evaluable_classes", "macro_auroc", "macro_auroc_ci_low", "macro_auroc_ci_high", "ece"]]
      .round(4).to_string(index=False))

def weighted_q(cal_scores, cal_w, test_w, alpha):
    """Per-test-point weighted conformal quantile (mass at +inf for the test point)."""
    if len(cal_scores) == 0:
        return np.full(len(test_w), np.inf)
    o = np.argsort(cal_scores)
    s, cw = cal_scores[o], np.cumsum(cal_w[o])
    target = (1 - alpha) * (cw[-1] + test_w)
    idx = np.searchsorted(cw, target, side="left")
    out = np.full(len(test_w), np.inf)
    ok = idx < len(s)
    out[ok] = s[idx[ok]]
    return out

def load_emb(split):
    return np.load(pred_path("resnet1d", emb_seed, split))["emb"].astype(np.float32)

EMB_CAL = load_emb("calib")
Pcal_src = PRED[PROPOSED]["calib"]["p_cal"]
ADAPT_ROWS, SHIFT_ROWS, TCAL_ROWS = [], [], []
t0 = time.time()
for key, d in EXT.items():
    Ye, labels = d["Y"], EVALUABLE[key]
    Pe = PRED[PROPOSED][f"ext_{key}"]["p_cal"]
    Ze = PRED[PROPOSED][f"ext_{key}"]["z"]
    Ee = load_emb(f"ext_{key}")
    N = len(Ye)
    n_adapt_list = [n for n in CFG["n_adapt"] if n <= 0.5 * N]
    n_max = max(n_adapt_list) if n_adapt_list else 0
    n_unl = min(CFG["n_unlabelled"], int(0.2 * N))
    if not labels or not n_adapt_list or N - n_max - n_unl < 50:
        log(f"{key}: too small for adaptation analysis (N={N}); skipped")
        continue
    for alpha in CFG["alphas"]:
        Q_src = QSTORE[(PROPOSED, "mondrian", alpha)]
        for r in range(CFG["n_repeats"]):
            rng = np.random.default_rng(10_000 + r)
            perm = rng.permutation(N)
            U = perm[n_max:n_max + n_unl]
            Ei = perm[n_max + n_unl:]
            # domain classifier source-calibration vs unlabelled target (M2/M3 weights)
            Zd = np.vstack([EMB_CAL, Ee[U]])
            dlab = np.r_[np.zeros(len(EMB_CAL)), np.ones(len(U))]
            clf = make_pipeline(StandardScaler(), PCA(n_components=min(32, Zd.shape[1], len(Zd) - 1), random_state=r),
                                LogisticRegression(C=1.0, max_iter=2000))
            p_oof = cross_val_predict(clf, Zd, dlab, cv=StratifiedKFold(5, shuffle=True, random_state=r),
                                      method="predict_proba")[:, 1]
            dom_auc = roc_auc_score(dlab, p_oof)
            clf.fit(Zd, dlab)
            ratio = len(EMB_CAL) / len(U)
            w_cal = np.clip(p_oof[: len(EMB_CAL)] / (1 - p_oof[: len(EMB_CAL)]) * ratio, 0.01, 100)
            pE = clf.predict_proba(Ee[Ei])[:, 1]
            w_E = np.clip(pE / (1 - pE) * ratio, 0.01, 100)
            SHIFT_ROWS.append({"site": key, "alpha": alpha, "repeat": r, "domain_auc": dom_auc,
                               "w_cal_clipped_frac": float(((w_cal <= 0.01) | (w_cal >= 100)).mean()),
                               "ess_frac_cal": float(w_cal.sum() ** 2 / (w_cal ** 2).sum() / len(w_cal))})
            PE = Pe[Ei]
            q_w = {}
            for j in labels:
                for c in (0, 1):
                    m = Ycal[:, j] == c
                    sc = (1 - Pcal_src[m, j]) if c == 1 else Pcal_src[m, j]
                    q_w[(j, c)] = weighted_q(sc, w_cal[m], w_E, alpha)
            for n in n_adapt_list:
                A = perm[:n]
                PA, YA = Pe[A], Ye[A]
                Q_tgt = mondrian_Q(PA, YA, alpha)
                Q_pool = mondrian_Q(np.vstack([Pcal_src, PA]), np.vstack([Ycal, YA]), alpha)
                for method in ["M0_source", "M1_target", "M2_weighted", "M3_hybrid", "M4_pooled"]:
                    q0 = np.zeros((len(Ei), len(CLASSES))); q1 = np.zeros_like(q0)
                    for j in labels:
                        for c, qarr in ((0, q0), (1, q1)):
                            if method == "M0_source":
                                qarr[:, j] = Q_src[j, c]
                            elif method == "M1_target":
                                qarr[:, j] = Q_tgt[j, c]
                            elif method == "M2_weighted":
                                qarr[:, j] = q_w[(j, c)]
                            elif method == "M3_hybrid":
                                qarr[:, j] = Q_tgt[j, c] if (YA[:, j] == c).sum() >= N_MIN_STRATUM_HYBRID else q_w[(j, c)]
                            else:
                                qarr[:, j] = Q_pool[j, c]
                    i0, i1 = cp_sets(PE, q0, q1)
                    ev = cp_eval(Ye[Ei], PE, i0, i1, labels, alpha)
                    rl = record_level(Ye[Ei], i0, i1, labels)
                    for _, e in ev.iterrows():
                        ADAPT_ROWS.append({"site": key, "alpha": alpha, "repeat": r, "n_adapt": n, "method": method,
                                           "class": e["class"], "true_class": e["true_class"], "n_eval": e["n"],
                                           "coverage": e["coverage"], "mean_set_size": e["mean_set_size"],
                                           "uncertain_rate": e["uncertain_rate"], "empty_rate": e["empty_rate"],
                                           "referral_rate_record": rl["referral_rate"],
                                           "exact_match_auto": rl["exact_match_auto"]})
                # calibration under shift: source temperature vs temperature refitted on A
                if alpha == CFG["alphas"][0]:
                    T_new = fit_temperature(Ze[A], Ye[A])
                    P_new = sigmoid(Ze[Ei] / T_new)
                    TCAL_ROWS.append({"site": key, "repeat": r, "n_adapt": n,
                                      "ece_source_T": ece_classwise(Ye[Ei][:, labels], PE[:, labels])[0],
                                      "ece_target_T": ece_classwise(Ye[Ei][:, labels], P_new[:, labels])[0]})
    log(f"{key}: adaptation analysis done ({time.time()-t0:.0f}s)")

T09_raw = pd.DataFrame(ADAPT_ROWS)
save_table(T09_raw, "S06_external_conformal_all_repeats")
if len(T09_raw):
    g = T09_raw.groupby(["site", "alpha", "n_adapt", "method", "class", "true_class"], as_index=False)
    T09 = g.agg(coverage_mean=("coverage", "mean"), coverage_sd=("coverage", "std"),
                coverage_p05=("coverage", lambda x: np.percentile(x, 5)), set_size_mean=("mean_set_size", "mean"),
                uncertain_rate_mean=("uncertain_rate", "mean"), referral_rate_mean=("referral_rate_record", "mean"),
                n_eval=("n_eval", "mean"), repeats=("repeat", "nunique"))
    T09["exploratory_mapping"] = T09["class"].map(lambda c: (not MAPPING_AUDIT_VALID) or not (KAPPA_BY_CLASS.get(c, 0) >= KAPPA_MIN_MAPPING))
else:
    T09 = pd.DataFrame()
save_table(T09, "T09_external_conformal_summary")
T10 = pd.DataFrame(SHIFT_ROWS).groupby(["site", "alpha"], as_index=False).agg(
    domain_auc_mean=("domain_auc", "mean"), domain_auc_sd=("domain_auc", "std"),
    ess_frac_mean=("ess_frac_cal", "mean"), clipped_frac_mean=("w_cal_clipped_frac", "mean")) if SHIFT_ROWS else pd.DataFrame()
save_table(T10, "T10_shift_magnitude")
T10b = (pd.DataFrame(TCAL_ROWS).groupby(["site", "n_adapt"], as_index=False)
        .agg(ece_source_T=("ece_source_T", "mean"), ece_target_T=("ece_target_T", "mean"),
             ece_target_T_sd=("ece_target_T", "std")) if TCAL_ROWS else pd.DataFrame())
save_table(T10b, "T10b_external_recalibration_ece")
if len(T09):
    show = T09[(T09.alpha == ALPHA_PRIMARY) & (T09.true_class == 1)]
    print(show.pivot_table(index=["site", "class", "n_adapt"], columns="method", values="coverage_mean").round(3).to_string())

## Robustness to acquisition artefacts (PTB-XL fold 10)
Perturbations applied to the preprocessed signal (mV) before normalisation: Gaussian noise at 20 and 10 dB SNR, baseline wander
(0.3 mV, 0.15-0.3 Hz), loss of one random lead (zeroed), and a 0.5x gain error. Source temperatures and Mondrian thresholds are
kept fixed, so coverage loss here measures sensitivity to artefacts, not a guaranteed quantity.

In [ ]:
def perturb(X, kind, rng):
    X = X.copy()
    if kind.startswith("noise"):
        snr = float(kind.split("_")[1])
        rms = np.sqrt((X ** 2).mean(axis=1, keepdims=True)) + 1e-6
        X += rng.normal(0, 1, X.shape).astype(np.float32) * rms / 10 ** (snr / 20)
    elif kind == "baseline_wander":
        t = np.arange(X.shape[1]) / TARGET_FS
        f = rng.uniform(0.15, 0.3, (X.shape[0], 1, 1)); ph = rng.uniform(0, 2 * np.pi, (X.shape[0], 1, X.shape[2]))
        X += (0.3 * np.sin(2 * np.pi * f * t[None, :, None] + ph)).astype(np.float32)
    elif kind == "lead_loss":
        lead = rng.integers(0, 12, X.shape[0])
        X[np.arange(X.shape[0]), :, lead] = 0.0
    elif kind == "gain_0.5":
        X *= 0.5
    return X

ROB_ROWS = []
if CFG["robustness"]:
    arch = "resnet1d"
    models_ = []
    for sd in CFG["seeds"]:
        m_ = ARCHS[arch]().to(DEVICE)
        m_.load_state_dict(torch.load(SUB["models"] / f"{arch}_seed{sd}_best.pt", map_location=DEVICE))
        models_.append(m_)
    Q = QSTORE[(PROPOSED, "mondrian", ALPHA_PRIMARY)]
    Xte = X_src[idx_test]
    for kind in ["clean", "noise_20", "noise_10", "baseline_wander", "lead_loss", "gain_0.5"]:
        Xp = Xte if kind == "clean" else perturb(Xte, kind, np.random.default_rng(GLOBAL_SEED))
        z = np.mean([predict(m_, Xp)[0] for m_ in models_], 0)
        P = sigmoid(z / PRED[PROPOSED]["T"])
        a0, a1 = cp_sets(P, Q[:, 0], Q[:, 1])
        ev = cp_eval(Yte, P, a0, a1)
        ROB_ROWS.append({"perturbation": kind, "macro_auroc": macro_auroc(Yte, P), "macro_ece": ece_classwise(Yte, P)[0],
                         "min_pos_coverage": float(ev[ev.true_class == 1].coverage.min()),
                         "mean_pos_coverage": float(ev[ev.true_class == 1].coverage.mean()),
                         "referral_rate": record_level(Yte, a0, a1)["referral_rate"]})
    del models_
T11 = pd.DataFrame(ROB_ROWS)
save_table(T11, "T11_robustness")
print(T11.round(4).to_string(index=False))

In [ ]:
# ---------- Efficiency ----------
EFF_ROWS = []
for arch in CFG["archs"]:
    m_ = ARCHS[arch]()
    m_.load_state_dict(torch.load(SUB["models"] / f"{arch}_seed{CFG['seeds'][0]}_best.pt", map_location="cpu"))
    n_par = sum(p.numel() for p in m_.parameters())
    size_mb = (SUB["models"] / f"{arch}_seed{CFG['seeds'][0]}_best.pt").stat().st_size / 1e6
    m_.eval(); x1 = torch.from_numpy(norm(X_src[idx_test[:1]])).permute(0, 2, 1)
    with torch.no_grad():
        for _ in range(5):
            m_(x1)
        ts = []
        for _ in range(50):
            t0 = time.perf_counter(); m_(x1); ts.append((time.perf_counter() - t0) * 1000)
    row = {"model": arch, "params_M": n_par / 1e6, "size_MB": size_mb, "cpu_ms_per_ecg_single": float(np.median(ts)),
           "cpu_ms_per_ecg_ensemble": float(np.median(ts)) * len(CFG["seeds"]), "cpu_threads": torch.get_num_threads()}
    if DEVICE.type == "cuda":
        mg = m_.to(DEVICE); xb = torch.from_numpy(norm(X_src[idx_test[:256]])).permute(0, 2, 1).to(DEVICE)
        with torch.no_grad(), torch.autocast("cuda", enabled=AMP):
            for _ in range(3):
                mg(xb)
            torch.cuda.synchronize(); t0 = time.perf_counter()
            for _ in range(10):
                mg(xb)
            torch.cuda.synchronize()
        row["gpu_ms_per_ecg_batch256"] = (time.perf_counter() - t0) * 1000 / (10 * len(xb))
    h = TRAIN_HIST[TRAIN_HIST.arch == arch]
    row.update(train_epoch_seconds_median=float(h.epoch_seconds.median()),
               epochs_run_mean=float(h.groupby("seed").epoch.max().mean()), hardware=RUNTIME_INFO["gpu"] or "CPU")
    EFF_ROWS.append(row)
T12 = pd.DataFrame(EFF_ROWS)
save_table(T12, "T12_efficiency")
print(T12.round(3).to_string(index=False))

## Figures (generated only from saved tables and predictions)

In [ ]:
OKABE = ["#0072B2", "#E69F00", "#009E73", "#D55E00", "#CC79A7", "#56B4E9", "#F0E442", "#000000"]
plt.rcParams.update({"font.size": 9, "axes.spines.top": False, "axes.spines.right": False, "font.family": "DejaVu Sans"})

# Fig 2: reliability diagrams, proposed model, before vs after temperature scaling
fig, axes = plt.subplots(1, len(CLASSES), figsize=(12, 2.8), sharey=True)
for j, (ax, c) in enumerate(zip(axes, CLASSES)):
    for kind, col, lab in [("p_raw", OKABE[1], "uncalibrated"), ("p_cal", OKABE[0], "temperature")]:
        p = PRED[PROPOSED]["test"][kind][:, j]; y = Yte[:, j]
        bins = np.array_split(np.argsort(p), 10)
        ax.plot([p[b].mean() for b in bins], [y[b].mean() for b in bins], "o-", ms=3, color=col, label=lab)
    ax.plot([0, 1], [0, 1], ":", color="grey", lw=1)
    ax.set_title(c); ax.set_xlabel("Predicted probability")
axes[0].set_ylabel("Observed frequency"); axes[0].legend(frameon=False, fontsize=7)
save_fig(fig, "Fig2_reliability_indomain")

# Fig 3: in-domain coverage per (class, true class), Mondrian vs global
t5 = pd.read_csv(SUB["tables"] / "T05_conformal_indomain.csv")
t5 = t5[(t5.model == PROPOSED) & (t5.alpha == ALPHA_PRIMARY)]
fig, ax = plt.subplots(figsize=(7, 3))
xlab = [f"{c}\n{'pos' if t == 1 else 'neg'}" for c in CLASSES for t in (1, 0)]
x = np.arange(len(xlab)); wdt = 0.38
for k, (variant, col) in enumerate([("mondrian", OKABE[0]), ("global", OKABE[1])]):
    sub_ = t5[t5.cp == variant].set_index(["class", "true_class"])
    cov = [sub_.loc[(c, t), "coverage"] for c in CLASSES for t in (1, 0)]
    lo = [sub_.loc[(c, t), "coverage"] - sub_.loc[(c, t), "cov_ci_low"] for c in CLASSES for t in (1, 0)]
    hi = [sub_.loc[(c, t), "cov_ci_high"] - sub_.loc[(c, t), "coverage"] for c in CLASSES for t in (1, 0)]
    ax.bar(x + (k - 0.5) * wdt, cov, wdt, yerr=[lo, hi], color=col, label=variant, capsize=2, error_kw={"lw": 0.8})
ax.axhline(1 - ALPHA_PRIMARY, color="black", ls="--", lw=1, label=f"target {1-ALPHA_PRIMARY:.2f}")
lowest = min(min(t5.cov_ci_low), 0.5)
ax.set_xticks(x); ax.set_xticklabels(xlab, fontsize=7); ax.set_ylim(max(0.0, np.floor(lowest * 10) / 10 - 0.05), 1.02); ax.set_ylabel("Coverage (95% CI)")
ax.legend(frameon=False, fontsize=7, ncol=3, loc="upper center", bbox_to_anchor=(0.5, 1.12))
save_fig(fig, "Fig3_coverage_mondrian_vs_global")

# Fig 4 and 5: external positive-class coverage and referral rate vs labelled budget
if len(T09):
    t9 = pd.read_csv(SUB["tables"] / "T09_external_conformal_summary.csv")
    t9 = t9[(t9.alpha == ALPHA_PRIMARY)]
    for figname, metric, ylab in [("Fig4_external_coverage_vs_budget", "coverage_mean", "Positive-class coverage"),
                                  ("Fig5_external_referral_vs_budget", "referral_rate_mean", "Record referral rate")]:
        sites = sorted(t9.site.unique())
        fig, axes = plt.subplots(1, len(sites), figsize=(5 * len(sites), 3), squeeze=False)
        for ax, site in zip(axes[0], sites):
            s = t9[(t9.site == site) & (t9.true_class == 1)]
            agg = s.groupby(["method", "n_adapt"], as_index=False)[metric].min() if metric == "coverage_mean" \
                else s.groupby(["method", "n_adapt"], as_index=False)[metric].mean()
            styles = ["-", "--", "-.", ":", (0, (5, 1))]; marks = ["o", "s", "^", "D", "v"]
            for k, (mth, grp) in enumerate(agg.groupby("method")):
                ax.plot(grp.n_adapt, grp[metric], linestyle=styles[k % 5], marker=marks[k % 5], color=OKABE[k], label=mth,
                        ms=5 - k * 0.6, lw=1.6, alpha=0.9)
            if metric == "coverage_mean":
                ax.axhline(1 - ALPHA_PRIMARY, color="black", ls="--", lw=1)
                ax.set_ylabel("Worst-class " + ylab.lower())
            else:
                ax.set_ylabel(ylab)
            ax.set_xscale("log"); ax.set_xlabel("Labelled target ECGs (n)"); ax.set_title(site)
        handles, labels_ = axes[0][0].get_legend_handles_labels()
        fig.legend(handles, labels_, frameon=False, fontsize=7, ncol=len(labels_), loc="upper center", bbox_to_anchor=(0.5, 1.06))
        save_fig(fig, figname)

# Fig 6: robustness
if len(T11):
    t11 = pd.read_csv(SUB["tables"] / "T11_robustness.csv")
    fig, ax1 = plt.subplots(figsize=(6, 3))
    xs = np.arange(len(t11))
    ax1.bar(xs - 0.2, t11.macro_auroc, 0.4, color=OKABE[0], label="macro-AUROC")
    ax1.bar(xs + 0.2, t11.min_pos_coverage, 0.4, color=OKABE[2], label="worst-class positive coverage")
    ax1.axhline(1 - ALPHA_PRIMARY, color="black", ls="--", lw=1)
    ax1.set_xticks(xs); ax1.set_xticklabels(t11.perturbation, rotation=20, fontsize=7); ax1.set_ylim(0.5, 1.0)
    ax1.legend(frameon=False, fontsize=7)
    save_fig(fig, "Fig6_robustness")

# Fig 7: subgroup positive coverage
if len(T06):
    t6 = pd.read_csv(SUB["tables"] / "T06_subgroup_coverage.csv")
    t6 = t6[t6.n_pos >= 30]
    fig, ax = plt.subplots(figsize=(7, 3))
    labels_ = [f"{r.dimension}:{r.group}" for r in t6.drop_duplicates(["dimension", "group"]).itertuples()]
    for k, c in enumerate(CLASSES):
        s = t6[t6["class"] == c]
        ax.plot([f"{r.dimension}:{r.group}" for r in s.itertuples()], s.pos_coverage, "o", color=OKABE[k], label=c)
    ax.axhline(1 - ALPHA_PRIMARY, color="black", ls="--", lw=1)
    ax.axhline(1 - ALPHA_PRIMARY - 0.05, color="grey", ls=":", lw=1)
    ax.set_ylabel("Positive-class coverage"); ax.tick_params(axis="x", rotation=30, labelsize=7); ax.legend(frameon=False, fontsize=7, ncol=5)
    save_fig(fig, "Fig7_subgroup_coverage")
# Fig 8 (post hoc): coverage across repeated calibration re-splits, alpha = 0.10
t15raw = pd.read_csv(SUB["tables"] / "S07_calibration_resplits_raw.csv")
t15raw = t15raw[t15raw.alpha == ALPHA_PRIMARY]
t15s = pd.read_csv(SUB["tables"] / "T15_calibration_resplits_summary.csv")
t15s = t15s[t15s.alpha == ALPHA_PRIMARY]
fig, ax = plt.subplots(figsize=(7, 3))
keys = [(c, t) for c in CLASSES for t in (1, 0)]
data = [t15raw[(t15raw["class"] == c) & (t15raw.true_class == t)].coverage.values for c, t in keys]
ax.boxplot(data, widths=0.5, showfliers=False, medianprops={"color": OKABE[0]})
origs = [t15s[(t15s["class"] == c) & (t15s.true_class == t)].original_split.values[0] for c, t in keys]
ax.plot(range(1, len(keys) + 1), origs, "D", color=OKABE[3], ms=4, label="primary split")
ax.axhline(1 - ALPHA_PRIMARY, color="black", ls="--", lw=1, label=f"target {1-ALPHA_PRIMARY:.2f}")
ax.set_xticks(range(1, len(keys) + 1)); ax.set_xticklabels([f"{c}\n{'pos' if t == 1 else 'neg'}" for c, t in keys], fontsize=7)
ax.set_ylabel("Test coverage across re-splits"); ax.legend(frameon=False, fontsize=7, loc="lower left")
save_fig(fig, "Fig8_coverage_calibration_resplits")

# Fig 9 (post hoc): positive-class coverage by age band, class-level vs age-aware Mondrian
t14 = pd.read_csv(SUB["tables"] / "T14_age_aware_conformal.csv")
t14 = t14[(t14.alpha == ALPHA_PRIMARY) & (t14.true_class == 1) & (t14.n >= 30)]
fig, ax = plt.subplots(figsize=(7.5, 3))
cells = t14[["age_band", "class"]].drop_duplicates().values.tolist()
xs = np.arange(len(cells)); wdt = 0.38
for k, (variant, col) in enumerate([("class_mondrian", OKABE[1]), ("age_aware_mondrian", OKABE[0])]):
    sub_ = t14[t14.variant == variant].set_index(["age_band", "class"])
    cov = [sub_.loc[(a, c), "coverage"] for a, c in cells]
    lo = [sub_.loc[(a, c), "coverage"] - sub_.loc[(a, c), "ci_low"] for a, c in cells]
    hi = [sub_.loc[(a, c), "ci_high"] - sub_.loc[(a, c), "coverage"] for a, c in cells]
    ax.bar(xs + (k - 0.5) * wdt, cov, wdt, yerr=[lo, hi], color=col, capsize=2, error_kw={"lw": 0.8},
           label=variant.replace("_", " "))
ax.axhline(1 - ALPHA_PRIMARY, color="black", ls="--", lw=1)
ax.set_xticks(xs); ax.set_xticklabels([f"{a}\n{c}" for a, c in cells], fontsize=6.5)
ax.set_ylim(0.4, 1.02); ax.set_ylabel("Positive-class coverage (95% CI)")
ax.legend(frameon=False, fontsize=7, ncol=2, loc="upper center", bbox_to_anchor=(0.5, 1.14))
save_fig(fig, "Fig9_age_aware_conformal")
print("Figures regenerated from saved tables.")

## Predeclared gates, verdict, and claim-to-evidence audit

In [ ]:
A = ALPHA_PRIMARY
gates = []
def gate(gid, name, criterion, observed, passed, evidence, critical):
    gates.append({"gate": gid, "name": name, "criterion": criterion, "observed": observed, "pass": bool(passed),
                  "critical": critical, "evidence": evidence})

t2 = T02.set_index("model")
prop_auc = float(t2.loc[PROPOSED, "macro_auroc"])
best_classical = max(["logreg", BASELINES[2]], key=lambda m: t2.loc[m, "macro_auroc"])
cmp_row = T04[T04.comparison == f"{PROPOSED} vs {best_classical}"].iloc[0]
gate("G1", "Discrimination", f"macro-AUROC >= 0.90 and > best classical ({best_classical}), Holm p < 0.05",
     f"{prop_auc:.4f}; delta {cmp_row.delta_macro_auroc:+.4f} [{cmp_row.ci_low:+.4f}, {cmp_row.ci_high:+.4f}], p_holm={cmp_row.p_holm:.4f}",
     prop_auc >= 0.90 and cmp_row.ci_low > 0 and cmp_row.p_holm < 0.05, "T02, T04", True)
t3 = T03[T03.model == PROPOSED].set_index("calibration")
gate("G2", "Calibration", "temperature-scaled macro ECE <= 0.03 and <= uncalibrated",
     f"{t3.loc['temperature','macro_ece']:.4f} vs {t3.loc['none','macro_ece']:.4f}",
     t3.loc["temperature", "macro_ece"] <= 0.03 and t3.loc["temperature", "macro_ece"] <= t3.loc["none", "macro_ece"], "T03", True)
m5 = T05[(T05.model == PROPOSED) & (T05.cp == "mondrian") & (T05.alpha == A)]
g5 = T05[(T05.model == PROPOSED) & (T05.cp == "global") & (T05.alpha == A)]
gate("G3", "In-domain stratum coverage", f"every (class, true class) coverage >= {1-A-0.02:.2f} (Mondrian)",
     f"min {m5.coverage.min():.3f} (global CP min {g5.coverage.min():.3f})", m5.coverage.min() >= 1 - A - 0.02, "T05", True)
sgp = T06[T06.n_pos >= 30] if len(T06) else T06
gate("G4", "Subgroup validity", f"positive coverage >= {1-A-0.05:.2f} in each sex/age group with >= 30 positives",
     f"min {sgp.pos_coverage.min():.3f} over {len(sgp)} cells" if len(sgp) else "no eligible cells",
     len(sgp) > 0 and sgp.pos_coverage.min() >= 1 - A - 0.05, "T06", False)
if len(T09):
    n_ref = 250 if 250 in set(T09.n_adapt) else int(T09.n_adapt.median())
    sel_ = T09[(T09.alpha == A) & (T09.true_class == 1) & (T09.n_adapt == n_ref)]
    m1 = sel_[sel_.method == "M1_target"].coverage_mean.min()
    m0 = sel_[sel_.method == "M0_source"].coverage_mean.min()
    gate("G5", "Shift restoration", f"M1 worst-class positive coverage at n={n_ref} >= {1-A-0.03:.2f} at every site",
         f"M1 {m1:.3f} (M0 source {m0:.3f})", m1 >= 1 - A - 0.03, "T09", True)
else:
    gate("G5", "Shift restoration", "external analysis available", "not run (sites too small)", False, "T09", True)
if len(T11):
    t11i = T11.set_index("perturbation")
    drop = t11i.loc["clean", "macro_auroc"] - t11i.loc["noise_20", "macro_auroc"]
    gate("G6", "Robustness", f"AUROC drop at 20 dB SNR <= 0.05 and positive coverage >= {1-A-0.05:.2f}",
         f"drop {drop:.4f}; coverage {t11i.loc['noise_20','min_pos_coverage']:.3f}",
         drop <= 0.05 and t11i.loc["noise_20", "min_pos_coverage"] >= 1 - A - 0.05, "T11", False)
t12 = T12.set_index("model")
gate("G7", "Compute", "ResNet1d CPU inference <= 100 ms per ECG (single model)",
     f"{t12.loc['resnet1d','cpu_ms_per_ecg_single']:.1f} ms", t12.loc["resnet1d", "cpu_ms_per_ecg_single"] <= 100, "T12", False)
gate("G0", "Data validity", "official PhysioNet source, all checksums verified, mapping audit valid",
     f"verified={DATA_VERIFIED}, mapping_audit_valid={MAPPING_AUDIT_VALID}", DATA_VERIFIED and MAPPING_AUDIT_VALID,
     "DATA_PROVENANCE.json, T01b", True)
GATES = pd.DataFrame(gates)
save_table(GATES, "T13_gate_dashboard")

crit_fail = GATES[GATES.critical & ~GATES["pass"]].gate.tolist()
noncrit_fail = GATES[~GATES.critical & ~GATES["pass"]].gate.tolist()
if MODE != "FULL_PAPER_RUN":
    VERDICT = f"{MODE}: NOT FOR PUBLICATION (gates shown for pipeline checking only)"
elif "G0" in crit_fail or "G3" in crit_fail:
    VERDICT = "STOP-REDESIGN"
elif crit_fail:
    VERDICT = "IMPROVE"
elif noncrit_fail:
    VERDICT = "GO-Q2-CANDIDATE"
else:
    VERDICT = "GO-Q1-CANDIDATE"
print(GATES[["gate", "name", "observed", "pass", "critical"]].to_string(index=False))
print(f"\nVERDICT: {VERDICT}  (critical failures: {crit_fail or 'none'}; other failures: {noncrit_fail or 'none'})")

In [ ]:
# ---------- RESULTS_AUDIT.md (claim-to-evidence) ----------
def fmt(v, d=3):
    return "n/a" if v is None or (isinstance(v, float) and not np.isfinite(v)) else f"{v:.{d}f}"

lines = [f"# RESULTS_AUDIT - {PROJECT_NAME}", "",
         f"- Mode: **{MODE}** | Manuscript eligible: **{MODE == 'FULL_PAPER_RUN' and DATA_VERIFIED}** | Verdict: **{VERDICT}**",
         f"- Run (UTC): {RUNTIME_INFO['timestamp_utc']} | Device: {RUNTIME_INFO['gpu'] or 'CPU'} | Seeds: {CFG['seeds']}",
         f"- Data: PTB-XL {PTBXL_VERSION} (n={len(meta_src)}, {meta_src.patient_id.nunique()} patients); "
         + "; ".join(f"{k} n={len(d['meta'])}" for k, d in EXT.items()),
         f"- Label-mapping ID alignment {fmt(id_alignment)}; audit valid: {MAPPING_AUDIT_VALID}", "",
         "## Frozen task definition",
         "- Task: multi-label classification of 5 PTB-XL diagnostic superclasses (NORM, MI, STTC, CD, HYP) from 10-s 12-lead ECGs at 100 Hz.",
         "- Splits: folds 1-8 train; fold 9 split by patient into tune (early stopping, temperature, thresholds, HPO) and calib "
         "(conformal only); fold 10 untouched test. External: Georgia, Chapman-Shaoxing (Challenge 2021 v1.0.3).",
         f"- Primary alpha: {A}; Mondrian stratification by label and true class; evaluable external classes need >= {CFG['min_pos_ext']} positives.",
         "", "## Claims and evidence", "| # | Claim | Value | Evidence | Gate | Status |", "|---|---|---|---|---|---|"]
claims = [
    ("C1", "Proposed macro-AUROC on PTB-XL fold 10", f"{fmt(prop_auc,4)} [{fmt(t2.loc[PROPOSED,'macro_auroc_ci_low'],4)}, {fmt(t2.loc[PROPOSED,'macro_auroc_ci_high'],4)}]", "T02", "G1"),
    ("C2", f"Gain over best classical baseline ({best_classical})", f"{cmp_row.delta_macro_auroc:+.4f} (Holm p={cmp_row.p_holm:.4f})", "T04", "G1"),
    ("C3", "Temperature scaling macro ECE (before -> after)", f"{fmt(t3.loc['none','macro_ece'],4)} -> {fmt(t3.loc['temperature','macro_ece'],4)}", "T03", "G2"),
    ("C4", "Mondrian worst stratum coverage in-domain", f"{fmt(m5.coverage.min())} (global CP {fmt(g5.coverage.min())})", "T05", "G3"),
    ("C5", "Worst subgroup positive coverage", fmt(sgp.pos_coverage.min()) if len(sgp) else "n/a", "T06", "G4"),
]
if len(T09):
    claims.append(("C6", f"External worst positive coverage at n={n_ref}: source vs target recalibration", f"{fmt(m0)} vs {fmt(m1)}", "T09", "G5"))
    claims.append(("C7", "Domain-classifier AUC (shift magnitude)", "; ".join(f"{r.site}: {fmt(r.domain_auc_mean)}" for r in T10.drop_duplicates("site").itertuples()), "T10", "-"))
if len(T11):
    claims.append(("C8", "AUROC drop at 20 dB SNR", fmt(drop, 4), "T11", "G6"))
claims.append(("C9", "ResNet1d CPU latency (ms/ECG)", fmt(t12.loc['resnet1d', 'cpu_ms_per_ecg_single'], 1), "T12", "G7"))
t16a = T16[T16.alpha == A]
claims.append(("C10", "[post hoc] Min Holm-adjusted p for in-domain under-coverage (exact finite-sample test)",
               fmt(float(t16a.p_holm.min())), "T16", "post hoc"))
t15a = T15[T15.alpha == A]
claims.append(("C11", "[post hoc] Mean worst-stratum coverage over calibration re-splits",
               f"{fmt(float(t15a['mean'].min()))} (per-stratum means {fmt(float(t15a['mean'].min()))}-{fmt(float(t15a['mean'].max()))})", "T15", "post hoc"))
t14p = T14[(T14.alpha == A) & (T14.true_class == 1) & (T14.n >= 30)]
if len(t14p):
    claims.append(("C12", "[post hoc] Worst age-band positive coverage: class-level vs age-aware Mondrian",
                   f"{fmt(float(t14p[t14p.variant=='class_mondrian'].coverage.min()))} vs {fmt(float(t14p[t14p.variant=='age_aware_mondrian'].coverage.min()))}",
                   "T14", "post hoc"))
gate_pass = GATES.set_index("gate")["pass"].to_dict()
for cid, text, val, ev, g in claims:
    status = ("exploratory - report as post hoc" if g == "post hoc" else
              "supported" if gate_pass.get(g, True) else "NOT supported - do not claim")
    lines.append(f"| {cid} | {text} | {val} | {ev} | {g} | {status} |")
lines += ["", "## Methods actually executed",
          "- Baselines: prevalence; logistic regression (C tuned on tune split) and HistGradientBoosting (leaf count tuned on tune split) on 184 features; FCN.",
          f"- Proposed: ResNet1d, {len(CFG['seeds'])}-seed ensemble, per-label temperature scaling, Mondrian split conformal.",
          "- External adaptation methods M0-M4 as defined in the notebook; ablations in T07; robustness in T11.",
          "- No synthetic data, no test-set tuning; conformal calibration split never used for fitting or model selection.",
          "", "## Missing or blocked items"]
blocked = []
if not len(T09):
    blocked.append("- External adaptation not run: sites too small. Blocks claim C6.")
if MODE != "FULL_PAPER_RUN":
    blocked.append(f"- Mode is {MODE}: no number here may appear in a manuscript. Re-run with MODE = 'FULL_PAPER_RUN'.")
if not MAPPING_AUDIT_VALID:
    blocked.append("- Mapping audit invalid: external per-class results are exploratory.")
lines += blocked or ["- none"]
lines += ["", "## Notes on the predeclared verdict",
          "- The verdict is computed by the rule declared before the first FULL run: a failed G3 maps to STOP-REDESIGN. "
          "That rule was intended to catch an invalid conformal implementation. The post hoc exact finite-sample check (T16) "
          "and repeated calibration re-splits (T15) test that question directly; report them as post hoc, and report G3 as "
          "not met at its predeclared threshold.",
          "- T14 (age-aware Mondrian) is a post hoc extension motivated by the G4 subgroup result."]
def md_table(df):
    cols = list(df.columns)
    out = ["| " + " | ".join(cols) + " |", "|" + "---|" * len(cols)]
    out += ["| " + " | ".join(str(v).replace("|", "/") for v in row) + " |" for row in df.itertuples(index=False)]
    return "\n".join(out)

lines += ["", "## Gate dashboard", md_table(GATES[["gate", "name", "criterion", "observed", "pass", "critical"]])]
audit_p = OUT / "RESULTS_AUDIT.md"
audit_p.write_text("\n".join(lines))
print(f"saved {audit_p.relative_to(OUT)} ({audit_p.stat().st_size/1024:.1f} KB)")

In [ ]:
# ---------- Run manifest, environment lock, output inventory, ZIP ----------
import importlib.metadata as ilm
env_lock = {}
for pkg in ["numpy", "pandas", "scipy", "scikit-learn", "torch", "matplotlib", "requests", "urllib3"]:
    try:
        env_lock[pkg] = ilm.version(pkg)
    except ilm.PackageNotFoundError:
        env_lock[pkg] = "not installed"
save_json({"python": sys.version, "packages": env_lock, "cuda": torch.version.cuda,
           "cudnn": torch.backends.cudnn.version() if torch.cuda.is_available() else None}, "config", "environment_lock")

EXPECTED_TABLES = ["T00_split_table", "T01_dataset_summary", "T01b_label_mapping_audit", "T02_discrimination_indomain",
                   "T03_calibration_indomain", "T04_paired_tests", "T05_conformal_indomain", "T05b_conformal_record_level",
                   "T06_subgroup_coverage", "T07_ablations", "T08_external_discrimination_calibration",
                   "T09_external_conformal_summary", "T10_shift_magnitude", "T10b_external_recalibration_ece",
                   "T11_robustness", "T12_efficiency", "T13_gate_dashboard", "T14_age_aware_conformal",
                   "T15_calibration_resplits_summary", "T16_finite_sample_coverage_check"]
inventory = []
for f in sorted(OUT.rglob("*")):
    if f.is_file() and "checkpoints" not in f.parts:
        inventory.append({"path": str(f.relative_to(OUT)), "bytes": f.stat().st_size})
inv = pd.DataFrame(inventory)
missing_tables = [t for t in EXPECTED_TABLES if not (SUB["tables"] / f"{t}.csv").exists()]
empty_files = inv[inv.bytes == 0].path.tolist()
manifest = {"project": PROJECT_NAME, "mode": MODE, "verdict": VERDICT, "manuscript_eligible": MODE == "FULL_PAPER_RUN" and DATA_VERIFIED,
            "runtime": RUNTIME_INFO, "config": CFG, "missing_tables": missing_tables, "empty_files": empty_files,
            "n_output_files": len(inv), "proposed_model": PROPOSED, "baselines": BASELINES}
save_json(manifest, "config", "run_manifest")
save_table(inv, "S99_output_inventory")
if missing_tables:
    print("Missing tables (and the claims they block):", missing_tables)
assert not empty_files, f"Empty output files: {empty_files}"

stamp = datetime.datetime.now(datetime.timezone.utc).replace(tzinfo=None).strftime("%Y%m%d_%H%M")
zip_path = WORK / f"{PROJECT_NAME}_results_{MODE}_{stamp}.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(OUT.rglob("*")):
        if f.is_file() and "checkpoints" not in f.parts:
            zf.write(f, f.relative_to(OUT.parent))
assert zip_path.exists() and zip_path.stat().st_size > 0
with zipfile.ZipFile(zip_path) as zf:
    assert zf.testzip() is None, "ZIP integrity check failed"
    n_zip = len(zf.namelist())
log(f"Results package: {zip_path} ({zip_path.stat().st_size/1e6:.1f} MB, {n_zip} files)")
if PERSIST:
    shutil.copyfile(zip_path, PERSIST / zip_path.name)
    log(f"Copy saved to Google Drive: {PERSIST / zip_path.name}")
if IN_COLAB:
    try:
        from google.colab import files
        files.download(str(zip_path))
    except Exception as e:
        print(f"Automatic download did not start ({e}). Download manually from the Files pane: {zip_path}")
print(f"\nDone. Mode {MODE}. Verdict: {VERDICT}. Upload {zip_path.name} to continue with the manuscript.")

## Limitations (to be carried into the manuscript in author voice)
- External labels come from a SNOMED CT to PTB-XL superclass mapping. Its agreement with PTB-XL's own labels is quantified
  (T01b) on the Challenge copy of PTB-XL, but cardiologist labelling conventions at Georgia and Chapman-Shaoxing may still differ.
- MI could not be evaluated externally (fewer than 100 positives at both sites); external conclusions cover NORM, STTC, CD and,
  where evaluable, HYP.
- The Challenge cohorts lack patient identifiers, so external resampling is record-level.
- Conformal guarantees are marginal within each (label, class) stratum under exchangeability; they are not conditional on an
  individual ECG, and the weighted variant (M2) is valid only under pure covariate shift.
- Signals were downsampled to 100 Hz and restricted to 10 s; higher-resolution morphology is not used.
- Robustness perturbations are simulated artefacts on real ECGs, not recordings from other devices.